<a href="https://colab.research.google.com/github/gurveervirk/CodeSage/blob/main/CodeSage_main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Install Dependencies
Note: last working version of gemini in llama-index for LATS agent is llama-index-llms-gemini==0.4.5

In [1]:
!pip install -q importlib typing llama-index-graph-stores-neo4j importlib-metadata astroid llama-index-llms-gemini llama-index-embeddings-gemini llama-index-embeddings-huggingface ratelimit aiolimiter flake8 llama-index-agent-lats llama-index-tools-duckduckgo

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.6/78.6 kB 1.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.3/275.3 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 312.3/312.3 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.7/62.7 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━

# COMStore (Code Object Model Store)

In [31]:
"""
This module is an integration of COM logic with the Neo4jPropertyGraphStore class.
"""
from llama_index.graph_stores.neo4j import Neo4jPropertyGraphStore
from llama_index.core.bridge.pydantic import PrivateAttr
from llama_index.core.graph_stores.types import (
    EntityNode, Relation
)
from llama_index.core import Settings
from typing import Any, Dict, List, Optional, Tuple
# from reflex import Entity, Relationship
import re
import logging
import neo4j

logging.basicConfig(level=logging.INFO)
BASE_ENTITY_LABEL = "__Entity__"
BASE_NODE_LABEL = "__Node__"

class Entity:
    """
    Module-agnostic entity class.
    """
    def __init__(self, name: str, label: str, properties: Dict):
        self.name = name
        self.label = label
        self.properties = properties

    def __repr__(self):
        return f"{self.label} ({self.name})"

class Relationship:
    """
    Module-agnostic relation class.
    """
    def __init__(self, from_entity: Entity, to_entity: Entity, label: str, properties: Dict):
        self.from_entity = from_entity
        self.to_entity = to_entity
        self.label = label
        self.properties = properties

    def __repr__(self):
        return f"{self.from_entity} --{self.label}--> {self.to_entity}"


def check_if_not_null(props: List[str], values: List[Any]) -> None:
    """Check if variable is not null and raise error accordingly."""
    for prop, value in zip(props, values):
        if not value:
            raise ValueError(f"Parameter `{prop}` must not be None or empty string")

FULL_TEXT_FULL_NAME_SEARCH_TEMPLATE = """
Could not find exact match...
Used a full-text index search to find the following best matches for the full paths(top 3 max):

{top_10_matches}
"""

class COMStore(Neo4jPropertyGraphStore):
    """
    COM stands for Code Object Model.
    This class is a subclass of Neo4jPropertyGraphStore and is used to store and retrieve code entities and their relationships in a Neo4j database.
    It provides ways to create, update, and query entities and relationships in the database. The database stores the Python Modular Tree structure in the form of nodes and relationships.

    Schema:
    - Nodes: MODULE, CLASS, METHOD
    - Relationships: HAS_CLASS, HAS_METHOD, INHERITS, IMPORTS
    """
    stores_text: bool = True
    flat_metadata: bool = True
    distance_strategy: str
    index_name: str
    keyword_index_name: str
    hybrid_search: bool
    node_label: str
    embedding_node_property: str
    text_node_property: str
    retrieval_query: str
    _support_metadata_filter: bool = PrivateAttr()
    _is_enterprise: bool = PrivateAttr()
    custom_fts_indices: Any = []

    _root_modules: List[Dict] = PrivateAttr()
    _children: List[Dict] = PrivateAttr()
    _root_modules_index: int = PrivateAttr()
    _children_index: int = PrivateAttr()
    _last_root_modules_query: Dict = PrivateAttr()
    _last_children_query: Dict = PrivateAttr()

    def __init__(
        self,
        username: str,
        password: str,
        url: str,
        database: Optional[str] = "neo4j",
        index_name: str = "entity",
        keyword_index_name: str = "keyword",
        node_label: str = "__Node__",
        embedding_node_property: str = "embedding",
        text_node_property: str = "source",
        retrieval_query: str = "",
        refresh_schema: bool = True,
        sanitize_query_output: bool = True,
        enhanced_schema: bool = False,
        **neo4j_kwargs: Any,
    ) -> None:
        self.index_name = index_name
        self.keyword_index_name = keyword_index_name
        self.node_label = node_label
        self.embedding_node_property = embedding_node_property
        self.text_node_property = text_node_property
        self.retrieval_query = retrieval_query
        self.database = database
        super().__init__(
            username,
            password,
            url,
            database,
            refresh_schema,
            sanitize_query_output,
            enhanced_schema,
            **neo4j_kwargs,
        )
        check_if_not_null(
            [
                "index_name",
                "node_label",
                "embedding_node_property",
                "text_node_property",
            ],
            [index_name, node_label, embedding_node_property, text_node_property],
        )
        self.create_required_indices()
        self._root_modules = []
        self._children = []
        self._root_modules_index = 0
        self._children_index = 0
        self._last_root_modules_query = {}
        self._last_children_query = {}

    def create_required_indices(self) -> None:
        """
        This method constructs Cypher queries and executes them to create:
        1. 5 full-text indices in Neo4j:
            - one for ROOT_MODULE entities on 'name' and 'source' properties.
            - one for MODULE entities on 'name' and 'source' properties.
            - one for FUNCTION entities on 'name' and 'source' properties.
            - one for CLASS entities on 'name' and 'source' properties.
            - one for METHOD entities on 'name' and 'source' properties.
        2. 5 vector indices:
            - one for ROOT_MODULE entities on the 'embedding' property.
            - one for FUNCTION entities on the 'embedding' property.
            - one for MODULE entities on the 'embedding' property.
            - one for CLASS entities on the 'embedding' property.
            - one for METHOD entities on the 'embedding' property.

        It uses the IF NOT EXISTS clause to avoid errors if the indices already exist.
        """

        # Create full-text indices
        root_module_index_query = """
        CREATE FULLTEXT INDEX root_module_index IF NOT EXISTS FOR (n:`ROOT_MODULE`) ON EACH [n.`name`]
        """
        self.database_query(root_module_index_query)

        function_index_query = """
        CREATE FULLTEXT INDEX function_index IF NOT EXISTS FOR (n:`FUNCTION`) ON EACH [n.`name`]
        """
        self.database_query(function_index_query)

        module_index_query = """
        CREATE FULLTEXT INDEX module_index IF NOT EXISTS FOR (n:`MODULE`) ON EACH [n.`name`]
        """
        self.database_query(module_index_query)

        class_index_query = """
        CREATE FULLTEXT INDEX class_index IF NOT EXISTS FOR (n:`CLASS`) ON EACH [n.`name`]
        """
        self.database_query(class_index_query)

        method_index_query = """
        CREATE FULLTEXT INDEX method_index IF NOT EXISTS FOR (n:`METHOD`) ON EACH [n.`name`]
        """
        self.database_query(method_index_query)

        # Create vector indices
        root_module_vector_index_query = """
        CREATE VECTOR INDEX root_module_vector_index IF NOT EXISTS
        FOR (m:`ROOT_MODULE`)
        ON (m.`embedding`)
        OPTIONS {indexConfig: {
            `vector.dimensions`: 1024,
            `vector.similarity_function`: 'cosine'
        }}
        """
        self.database_query(root_module_vector_index_query)

        function_vector_index_query = """
        CREATE VECTOR INDEX function_vector_index IF NOT EXISTS
        FOR (f:`FUNCTION`)
        ON (f.`embedding`)
        OPTIONS {indexConfig: {
            `vector.dimensions`: 1024,
            `vector.similarity_function`: 'cosine'
        }}
        """
        self.database_query(function_vector_index_query)

        module_vector_index_query = """
        CREATE VECTOR INDEX module_vector_index IF NOT EXISTS
        FOR (m:`MODULE`)
        ON (m.`embedding`)
        OPTIONS {indexConfig: {
            `vector.dimensions`: 1024,
            `vector.similarity_function`: 'cosine'
        }}
        """
        self.database_query(module_vector_index_query)

        class_vector_index_query = """
        CREATE VECTOR INDEX class_vector_index IF NOT EXISTS
        FOR (c:`CLASS`)
        ON (c.`embedding`)
        OPTIONS {indexConfig: {
            `vector.dimensions`: 1024,
            `vector.similarity_function`: 'cosine'
        }}
        """
        self.database_query(class_vector_index_query)

        method_vector_index_query = """
        CREATE VECTOR INDEX method_vector_index IF NOT EXISTS
        FOR (m:`METHOD`)
        ON (m.`embedding`)
        OPTIONS {indexConfig: {
            `vector.dimensions`: 1024,
            `vector.similarity_function`: 'cosine'
        }}
        """
        self.database_query(method_vector_index_query)

        general_fts_query = """
        CREATE FULLTEXT INDEX general_fts_index IF NOT EXISTS FOR (n:__Entity__) ON EACH [n.name]
        """
        self.database_query(general_fts_query)

    def retrieve_relevant_root_modules(self, query: str, top_k: int = 10) -> str:
        """
        Retrieve all relevant root modules based on the query, using similarity search, and store them.

        Args:
            query (str): The query string.
            top_k (int): The number of results to return. Defaults to 10.

        Returns:
            str: Top k relevant root modules based on the query formatted as a string.
        """
        # Clean input params
        top_k = int(top_k)

        # Create a Cypher query to return all root modules based on similarity to the query
        retrieve_root_modules_query = """
            MATCH (m:ROOT_MODULE)
            WHERE m.embedding IS NOT NULL
            WITH m,
                vector.similarity.cosine(m.embedding, $query_vector) AS similarity
            ORDER BY similarity DESC
            RETURN m.name AS name, similarity AS score, m.summary AS summary
        """

        query_params = {
            "query_vector": Settings.embed_model.get_query_embedding(query)
        }

        # Execute the query to retrieve all relevant root modules
        results = self.database_query(retrieve_root_modules_query, query_params)

        # Store the results
        self._root_modules = results
        self._root_modules_index = top_k
        self._last_root_modules_query = {"query": query, "top_k": top_k}

        # Format the top k results as a string
        formatted_results = "Relevant Root Modules:\n\n" + "\n\n".join(
            [f"{i+1}. **{result['name']}** (similarity score: {result['score']}):{result['summary']}" for i, result in enumerate(self._root_modules[:top_k])]
        )

        return formatted_results

    def retrieve_relevant_children_in_subtree(self, query: str, parent_name: str, top_k: int = 10) -> str:
        """
        Retrieve all relevant IMMEDIATE children in the subtree of the given parent based on the query, using similarity search, and store them.

        Args:
            query (str): The query string.
            parent_name (str): The name of the parent entity.
            top_k (int): The number of results to return. Defaults to 10.

        Returns:
            str: Top k Relevant children in the subtree of the parent based on the query formatted as a string.
        """
        # Clean input params
        top_k = int(top_k)

        # Embed the query
        query_vector = Settings.embed_model.get_query_embedding(query)

        # Use a cypher query to return all IMMEDIATE children of the parent entity, sorted by similarity to the query
        sort_and_return_children_query = """
            MATCH (parent {name: $parent_name})-[:HAS_SUBMODULE|HAS_CLASS|HAS_METHOD|HAS_FUNCTION|EXPOSES]->(child)
            WHERE child.embedding IS NOT NULL
            WITH parent, child,
                vector.similarity.cosine(child.embedding, $query_vector) AS similarity
            ORDER BY similarity DESC
            RETURN child.name AS name, similarity AS score, labels(child) AS labels, child.summary AS summary
        """

        query_params = {
            "parent_name": parent_name,
            "query_vector": query_vector,
        }

        try:
            records = self.database_query(sort_and_return_children_query, query_params)

            # Filter out base labels and retain only the relevant labels
            for record in records:
                record["labels"] = [label for label in record["labels"] if label not in [BASE_ENTITY_LABEL, BASE_NODE_LABEL]]

            # Store the results
            self._children = records
            self._children_index = top_k
            self._last_children_query = {"query": query, "parent_name": parent_name, "top_k": top_k}

            # Format the top k results as a string
            formatted_results = "Relevant Children in Subtree:\n\n" + "\n\n".join(
                [f"{i+1}. **{result['name']}** (similarity score: {result['score']}, relevant labels: {result['labels']}):\n{result['summary']}" for i, result in enumerate(self._children[:top_k])]
            )

            return formatted_results

        except Exception as e:
            logging.error(f"Error retrieving relevant children in subtree: {e}")
            return []

    def next_entities(self, from_: str, top_k: int = 10) -> str:
        """
        Retrieve the next top_k entities from the stored list.

        Args:
            from_ (str): "ROOT_MODULE" or "CHILDREN", indicating which list to retrieve from.
            top_k (int): The number of entities to retrieve. Defaults to 10.

        Returns:
            str: Formatted string of the next top_k entities.
        """
        top_k = int(top_k)
        if from_ == "ROOT_MODULE":
            if self._root_modules == None or self._root_modules_index == -1:
                return "NO ROOT MODULES HAVE BEEN QUERIED FOR. PROCEED WITH QUERYING `retrieve_relevant_root_modules`."
            start_index = self._root_modules_index
            end_index = min(start_index + top_k, len(self._root_modules))
            entities = self._root_modules[start_index:end_index]
            self._root_modules_index = end_index
            formatted_results = "Next Relevant Root Modules:\n\n" + "\n\n".join(
                [f"{i+1}. **{result['name']}** (similarity score: {result['score']}):{result['summary']}" for i, result in enumerate(entities)]
            )
            if end_index == len(self._root_modules):
                formatted_results += "\n\nALL RELEVANT ROOT MODULES RETRIEVED. PROCEED WITH ANOTHER QUERY USING `retrieve_relevant_root_modules`."
                self._root_modules = None
                self._root_modules_index = -1

        elif from_ == "CHILDREN":
            if self._children == None or self._children_index == -1:
                return "NO CHILDREN HAVE BEEN QUERIED FOR. PROCEED WITH QUERYING `retrieve_relevant_children_in_subtree`."
            start_index = self._children_index
            end_index = min(start_index + top_k, len(self._children))
            entities = self._children[start_index:end_index]
            self._children_index = end_index
            formatted_results = "Next Relevant Children in Subtree:\n\n" + "\n\n".join(
                [f"{i+1}. **{result['name']}** (similarity score: {result['score']}, relevant labels: {result['labels']}):\n{result['summary']}" for i, result in enumerate(entities)]
            )
            if end_index == len(self._root_modules):
                formatted_results += "\n\nALL RELEVANT CHILDREN RETRIEVED. PROCEED WITH ANOTHER QUERY USING `retrieve_relevant_children_in_subtree`."
                self._children = None
                self._children_index = -1
        else:
            raise ValueError("`from` must be either 'ROOT_MODULE' or 'CHILDREN'")

        return formatted_results

    def query_vector_index(
        self, index_name: str, query: str | List[float], top_k: int = 10,
        **kwargs: Any
    ) -> List[Dict]:
        """
        Query a vector index.

        Args:
            index_name (str): The name of the vector index, restricted to `method_vector_index`, `class_vector_index` and `module_vector_index`.
            query (str or List[float]): The query or query vector embedding.
            top_k (int): The number of results to return. Defaults to 10.

        Returns:
            List[Dict]: A list of dictionaries containing the query results.
        """
        query_vector = None

        if isinstance(query, str):
            query_vector = Settings.embed_model.get_query_embedding(query)
        else:
            query_vector = query

        return_fields = ["name", "summary", "source"]
        return_fields_str = ", ".join([f"n.{field} AS {field}" for field in return_fields])

        query_str = f"""
        CALL db.index.vector.queryNodes($index_name, $top_k, $query_vector)
        YIELD node AS n, score
        RETURN {return_fields_str}, score, labels(n) as labels
        """
        query_params = {
            "index_name": index_name,
            "top_k": top_k,
            "query_vector": query_vector,
        }

        try:
            records = self.database_query(query_str, query_params)
            return records
        except Exception as e:
            logging.error(f"Error querying vector index '{index_name}': {e}")
            return []

    def query_fulltext_index(
        self, index_name: str, query: str,
        **kwargs: Any
    ) -> List[Dict]:
        """
        Query a full-text index.

        Args:
            index_name (str): The name of the full-text index, restricted to `module_index`, `class_index` and `method_index`.
            query (str): The query string.
        Returns:
            List[Dict]: A list of dictionaries containing the query results.
        """
        return_fields = ["name", "source"]
        return_fields_str = ", ".join([f"n.{field} AS {field}" for field in return_fields])

        query_str = f"""
        CALL db.index.fulltext.queryNodes($index_name, $query)
        YIELD node AS n, score
        RETURN {return_fields_str}, score
        """
        query_params = {
            "index_name": index_name,
            "query": query,
        }

        try:
            records = self.database_query(query_str, query_params)
            return records
        except Exception as e:
            logging.error(f"Error querying full-text index '{index_name}': {e}")
            return []

    def similarity_search_without_index(
        self,
        labels: List[str],
        query: str | List[float],
        top_k: int = 10,
        **kwargs: Any,
    ) -> List[Dict]:
        """
        Perform a similarity search without using an index.

        Args:
            labels (List[str]): A list of labels to query (e.g., ["MODULE", "CLASS", "METHOD"]).
            query (str or List[float]): The query or query vector embedding.
            top_k (int): The number of results to return. Defaults to 10.

        Returns:
            List[Dict]: A list of dictionaries containing the query results.
        """
        query_vector = None
        if isinstance(query, str):
            query_vector = Settings.embed_model.get_query_embedding(query)
        else:
            query_vector = query
        return_fields = ["name", "source"]
        return_fields_str = ", ".join([f"n.{field} AS {field}" for field in return_fields])

        query_str = f"""
        MATCH (n)
        WHERE any(label in labels(n) WHERE label IN $labels)
        AND n.{self.embedding_node_property} IS NOT NULL
        WITH n,
        vector.similarity.cosine(n.{self.embedding_node_property}, $query_vector) AS similarity
        ORDER BY similarity DESC
        LIMIT $top_k
        RETURN {return_fields_str}, labels(n) AS all_labels, similarity AS score
        """

        query_params = {
            "labels": labels,
            "query_vector": query_vector,
            "top_k": top_k,
        }

        try:
            records = self.database_query(query_str, query_params)
            return records
        except Exception as e:
            logging.error(f"Error performing similarity search: {e}")
            return []

    def database_query(
        self,
        query: str,
        params: Optional[Dict[str, Any]] = None,
    ) -> Any:
        """
        Execute a cypher query in the Neo4j database.

        Args:
            query (str): The query string.
            params (Optional[Dict[str, Any]]): A dictionary of parameters to pass to the query.
        """
        params = params or {}
        try:
            data, _, _ = self._driver.execute_query(
                query, database_=self.database, parameters_=params
            )
            return [r.data() for r in data]
        except neo4j.exceptions.Neo4jError as e:
            if not (
                (
                    (  # isCallInTransactionError
                        e.code == "Neo.DatabaseError.Statement.ExecutionFailed"
                        or e.code
                        == "Neo.DatabaseError.Transaction.TransactionStartFailed"
                    )
                    and "in an implicit transaction" in e.message
                )
                or (  # isPeriodicCommitError
                    e.code == "Neo.ClientError.Statement.SemanticError"
                    and (
                        "in an open transaction is not possible" in e.message
                        or "tried to execute in an explicit transaction" in e.message
                    )
                )
            ):
                raise
        # Fallback to allow implicit transactions
        with self._driver.session(database=self.database) as session:
            data = session.run(neo4j.Query(text=query), params)
            return [r.data() for r in data]

    def upsert_to_neo4j(
            self,
            module_name: str,
            entities: List[Entity],
            relations: List[Relationship],
            other_relations: Dict[str, List[Tuple]],
            embed_nodes: Optional[bool] = False
        ) -> None:
        """
        Upsert entities and relationships into Neo4j.

        Args:
            module_name (str): The name of the module.
            entities (List[Entity]): List of entity nodes to upsert.
            relations (List[Relationship]): List of relationships to upsert.
            other_relations (Dict[str, List[Tuple]]): Dictionary containing other relations.
        """
        # Convert Entity to EntityNode and Relationship to Relation
        entity_nodes = []
        entities_to_entity_nodes = {}

        for entity in entities:
            entity_node = EntityNode(
                label=entity.label,
                name=entity.name,
                properties=entity.properties
            )
            entity_nodes.append(entity_node)
            entities_to_entity_nodes[entity.name] = entity_node

        relations = [
            Relation(
                label=r.label,
                source_id=entities_to_entity_nodes[r.from_entity.name].id,
                target_id=entities_to_entity_nodes[r.to_entity.name].id,
                properties=r.properties
            ) for r in relations
        ]

        self.upsert_nodes(entity_nodes)
        self.upsert_relations(relations)

        # Embed the nodes if specified
        if embed_nodes:
            self._embed_nodes()

        # Add INHERITS and IMPORTS relations as properties to the respective nodes
        self._add_other_relations_as_properties(other_relations)

        # Process INHERITS and IMPORTS relations
        self._process_other_relations()

        # Set the shortest call paths for all nodes in the module
        self._set_shortest_call_paths(module_name)

        # Add special labels to entities starting with '_'
        self._add_special_labels(module_name)

        # Add labels recursively
        self._handle_inherited_methods()

    def _add_special_labels(self, module_name: str):
        """
        Add internal labels to entities starting with '_' in the specified module's subtree.

        Args:
            module_name (str): The name of the module.
        """
        # Create cypher queries to add special labels to entities starting with '_'
        add_dunder_labels_query = (
            """
            MATCH (root:ROOT_MODULE {name: $module_name})-[:HAS_SUBMODULE|HAS_CLASS|HAS_METHOD|HAS_FUNCTION|EXPOSES*]->(children:FUNCTION|METHOD)
            WHERE children.__name__ STARTS WITH '__' AND children.__name__ ENDS WITH '__'
            CALL apoc.create.addLabels(children, ['DUNDER'])
            YIELD node
            RETURN count(node) AS numNodesLabeled
            """
        )
        add_private_labels_query = (
            """
            MATCH (root:ROOT_MODULE {name: $module_name})-[:HAS_SUBMODULE|HAS_CLASS|HAS_METHOD|HAS_FUNCTION|EXPOSES*]->(children)
            WHERE children.__name__ STARTS WITH '__' AND NOT children.__name__ ENDS WITH '__'
            CALL apoc.create.addLabels(children, ['PRIVATE'])
            YIELD node
            RETURN count(node) AS numNodesLabeled
            """
        )
        add_internal_labels_query = (
            """
            MATCH (root:ROOT_MODULE {name: $module_name})-[:HAS_SUBMODULE|HAS_CLASS|HAS_METHOD|HAS_FUNCTION|EXPOSES*]->(children)
            WHERE NOT children.__name__ STARTS WITH '__' AND (children.__name__ STARTS WITH '_' OR children.shortest_call_path CONTAINS '._')
            CALL apoc.create.addLabels(children, ['INTERNAL'])
            YIELD node
            RETURN count(node) AS numNodesLabeled
            """
        )

        self.structured_query(add_dunder_labels_query, {"module_name": module_name})
        self.structured_query(add_private_labels_query, {"module_name": module_name})
        self.structured_query(add_internal_labels_query, {"module_name": module_name})

    def _add_other_relations_as_properties(self, other_relations: Dict[str, List[Tuple]]):
        """
        Add INHERITS and IMPORTS relations as properties to the respective nodes.

        Args:
            other_relations (Dict[str, List[Tuple]]): Dictionary containing INHERITS and IMPORTS relations.
        """
        inherits_relations = other_relations.get("INHERITS", [])
        imports_relations = other_relations.get("IMPORTS", [])
        exposes_relations = other_relations.get("EXPOSES", [])
        add_inherits_query = (
            """
            UNWIND $inherits_relations AS rel
            MATCH (c:CLASS {name: rel[0]})
            SET c.INHERITS = apoc.coll.toSet(coalesce(c.INHERITS, []) + rel[1])
            """
        )
        add_imports_query = (
            """
            UNWIND $imports_relations AS rel
            MATCH (e {name: rel[0]})
            SET e.IMPORTS = apoc.coll.toSet(coalesce(e.IMPORTS, []) + rel[1])
            """
        )
        add_exposes_query = (
            """
            UNWIND $exposes_relations AS rel
            MATCH (m {name: rel[0]})
            SET m.EXPOSES = apoc.coll.toSet(coalesce(m.EXPOSES, []) + rel[1])
            """
        )
        self.structured_query(add_inherits_query, {"inherits_relations": inherits_relations})
        self.structured_query(add_imports_query, {"imports_relations": imports_relations})
        self.structured_query(add_exposes_query, {"exposes_relations": exposes_relations})

    def _process_other_relations(self):
        """
        Execute Cypher queries to create relations on existing nodes with INHERITS, IMPORTS and EXPOSES properties.
        Clear names from the INHERITS, IMPORTS and EXPOSES property arrays of the nodes if the relations could be created successfully.
        """
        process_inherits_query = (
            """
            MATCH (c:CLASS)
            WHERE c.INHERITS IS NOT NULL
            UNWIND c.INHERITS AS parent
            MATCH (p:CLASS {name: parent})
            MERGE (c)-[:INHERITS]->(p)
            WITH c, COLLECT(parent) AS parentNames
            SET c.INHERITS = [name IN c.INHERITS WHERE NOT name IN parentNames]
            SET c.NEW_INHERIT_RELS = parentNames
            """
        )
        process_imports_query = (
            """
            MATCH (e1)
            WHERE e1.IMPORTS IS NOT NULL
            UNWIND e1.IMPORTS AS importee
            MATCH (e2 {name: importee})
            MERGE (e1)-[:IMPORTS]->(e2)
            WITH e1, COLLECT(importee) AS importeeNames
            SET e1.IMPORTS = [name IN e1.IMPORTS WHERE NOT name IN importeeNames]
            """
        )
        process_exposes_query = (
            """
            MATCH (m:MODULE|ROOT_MODULE)
            WHERE m.EXPOSES IS NOT NULL
            UNWIND m.EXPOSES AS exposed
            MATCH (e {name: exposed})
            MERGE (m)-[:EXPOSES]->(e)
            WITH m, COLLECT(exposed) AS exposedNames
            SET m.EXPOSES = [name IN m.EXPOSES WHERE NOT name IN exposedNames]
            """
        )
        cleanup_empty_INHERITS_query = (
            """
            MATCH (n)
            WHERE n.INHERITS IS NOT NULL AND size(n.INHERITS) = 0
            REMOVE n.INHERITS
            """
        )
        cleanup_empty_IMPORTS_query = (
            """
            MATCH (n)
            WHERE n.IMPORTS IS NOT NULL AND size(n.IMPORTS) = 0
            REMOVE n.IMPORTS
            """
        )
        cleanup_empty_EXPOSES_query = (
            """
            MATCH (n)
            WHERE n.EXPOSES IS NOT NULL AND size(n.EXPOSES) = 0
            REMOVE n.EXPOSES
            """
        )
        add_names_to_EXPOSES_rels_query = (
            """
            MATCH (n)-[r:EXPOSES]->(m)
            WHERE r.name IS NULL
            SET r.name = n.name + "." + m.__name__
            RETURN count(r)
            """
        )

        self.structured_query(process_inherits_query)
        self.structured_query(process_imports_query)
        self.structured_query(process_exposes_query)
        self.structured_query(cleanup_empty_INHERITS_query)
        self.structured_query(cleanup_empty_IMPORTS_query)
        self.structured_query(cleanup_empty_EXPOSES_query)
        self.structured_query(add_names_to_EXPOSES_rels_query)

    def _set_shortest_call_paths(self, module_name: str):
        """
        Set the shortest call paths for all nodes in the module.

        Args:
            module_name (str): The name of the module.
        """
        # Set the shortest call paths for all nodes in the module tree (recursively)
        set_shortest_call_paths_query = (
            """
            MATCH (end:MODULE|FUNCTION|CLASS|METHOD)<-[:HAS_SUBMODULE|HAS_CLASS|HAS_METHOD|HAS_FUNCTION|EXPOSES*]-(root:ROOT_MODULE {name: $module_name})
            WITH root, collect(end) AS endNodes
            CALL apoc.path.spanningTree(root, {
              relationshipFilter: 'HAS_SUBMODULE>|HAS_CLASS>|HAS_METHOD>|HAS_FUNCTION>|EXPOSES>',
              endNodes: endNodes
            })
            YIELD path
            WITH nodes(path) AS nodesInPath
            WITH LAST(nodesInPath) AS targetNode, nodesInPath[-2] as parentNode
            WHERE targetNode.shortest_call_path IS NULL
            SET targetNode.shortest_call_path =
            CASE
                WHEN size(parentNode.name + "." + targetNode.__name__) < size(targetNode.name)
                THEN parentNode.name + "." + targetNode.__name__
                ELSE targetNode.name
            END
            RETURN count(targetNode) AS numPathsSet
            """
        )
        numPathsSet = self.structured_query(set_shortest_call_paths_query, {"module_name": module_name})[0]["numPathsSet"]
        while numPathsSet > 0:
            numPathsSet = self.structured_query(set_shortest_call_paths_query, {"module_name": module_name})[0]["numPathsSet"]

    def _handle_inherited_methods(self):
        """
        Create HAS_METHOD relations for inherited methods for all classes in the specified module.

        Args:
            module_name (str): The name of the module.

        """
        handle_inheritance_query = (
            """
            MATCH (child:CLASS)
            WHERE child.NEW_INHERIT_RELS IS NOT NULL
            WITH child, child.NEW_INHERIT_RELS AS parentNames
            REMOVE child.NEW_INHERIT_RELS
            WITH child, parentNames
            UNWIND parentNames AS parentName
            MATCH (parent:CLASS {name: parentName})-[:HAS_METHOD]->(member)
            WHERE NOT EXISTS {
                MATCH (child)-[:HAS_METHOD]->(childMethod)
                WHERE childMethod.__name__ = member.__name__ AND childMethod.signature = member.signature
            }
            WITH child, collect(DISTINCT member) AS members
            UNWIND members AS member
            MERGE (child)-[:HAS_METHOD]->(member)
            WITH DISTINCT child, collect(member) AS members
            RETURN child.name AS ChildClass, size(members) AS TotalParentMembersLabeled
            """
        )
        self.structured_query(handle_inheritance_query)

    def _remove_empty_values(self, input_dict):
        """
        Remove entries with empty values from the dictionary.

        Parameters:
        input_dict (dict): The dictionary from which empty values need to be removed.

        Returns:
        dict: A new dictionary with all empty values removed.
        """
        # Create a new dictionary excluding empty values
        return {key: value for key, value in input_dict.items() if value}

    def _embed_nodes(self):
        """
        Embed newly created nodes.
        """
        retrieve_new_nodes = (
            """
            MATCH (e)
            WHERE NOT e:INTERNAL
            AND NOT e:PRIVATE
            AND NOT e:DUNDER
            AND e.embedding IS NULL
            RETURN
            e.id AS name,
            labels(e)[0] AS type,
            properties(e) AS properties
            """
        )
        new_nodes = self.structured_query(retrieve_new_nodes)
        if not new_nodes:
            return

        logging.info(f"Found {len(new_nodes)} new nodes to embed.")

        entity_nodes, summaries = [], []
        # Create EntityNode objects for each node
        for node in new_nodes:
            entity_node = EntityNode(
                label=node["type"],
                name=node["name"],
                properties=self._remove_empty_values(node["properties"])
            )
            entity_nodes.append(entity_node)
            summary = entity_node.properties.get("summary", "")

            if node["type"] in ("ROOT_MODULE", "MODULE", "CLASS"):
                docstring = self._get_docstring(entity_node.properties.get("source", ""))
                if docstring is not None and len(docstring) > 0:
                    summary += f"\n\nDocstring:\n{docstring}"

            summaries.append(summary)

        # Embed the nodes
        embeddings = Settings.embed_model.get_text_embedding_batch(
            summaries,
            show_progress=True
        )

        for i, node in enumerate(entity_nodes):
            node.properties[self.embedding_node_property] = embeddings[i]

        self.upsert_nodes(entity_nodes)

    def _get_entity_info(self, entity_name: str) -> List[str]:
        """
        Get information about the entity.

        Args:
            entity_name (str): The name of the entity.

        Returns:
            List[str]: Information about the entity.
        """
        get_entity_info_query = (
            """
            MATCH (end {name: $entity_name})
            RETURN end.name AS name, end.__name__ AS __name__, end.summary AS summary, end.signature AS signature, end.llm_signature AS llm_signature, end.shortest_call_path AS shortest_call_path
            """
        )
        result = self.structured_query(get_entity_info_query, {"entity_name": entity_name})
        return result if result else []

    def _get_retrieved_entity_representation(self, retrieved_node: Dict) -> str:
        """
        Get the representation of the retrieved entity.

        Args:
            retrieved_node (Dict): The retrieved node.

        Returns:
            str: The representation of the retrieved entity.
        """
        entity_representation = f"""
        *Name*: {retrieved_node["__name__"]}
        *Full Path*: {retrieved_node["name"]}
        *Shortest Call Path*: {retrieved_node["shortest_call_path"]}
        *Signature*: {retrieved_node["signature"]}
        *Signature identified by LLM as JSON (ONLY TO BE USED IF TYPES CANNOT BE ASCERTAINED FROM THE `Signature` FIELD)*:
        {retrieved_node["llm_signature"]}

        *Summary of the Entity*:
        ```
        {retrieved_node["summary"]}
        ```
        """
        return entity_representation

    def get_entity_info(self, entity_name: str) -> str:
        """
        Get information about the entity.

        Args:
            entity_name (str): The name of the entity.

        Returns:
            str: Information about the entity.
        """
        entity_node = self._get_entity_info(entity_name)
        if not entity_node:
            return "No information found for the entity."

        entity_representation = self._get_retrieved_entity_representation(entity_node[0])
        return entity_representation

    def get_full_path_name(self, name: str) -> str:
        """
        Get the full path name of the entity.

        Args:
            name (str): The name of the entity.

        Returns:
            str: The full path name of the entity.
        """
        retrieve_node_by_name_query = (
            """
            MATCH (n:METHOD|FUNCTION|CLASS|MODULE {name: $name})
            RETURN n.name AS name
            """
        )

        # Execute the query to retrieve the node by name
        retrieved_node = self.structured_query(retrieve_node_by_name_query, {"name": name})

        if not retrieved_node:
            # If no relevant nodes are found, try querying relationships with this name
            retrieve_relationship_by_name_query = (
                """
                MATCH (n)-[r:EXPOSES]->(m)
                WHERE r.name = $name
                RETURN m.name AS name
                """
            )

            # Execute the query to retrieve relevant relationships
            retrieved_node = self.structured_query(retrieve_relationship_by_name_query, {"name": name})

        if retrieved_node:
            name = retrieved_node[0]["name"]
            return name

        # Clean the name for FTS
        name = name.replace(".", "*")
        records = self.query_fulltext_index("general_fts_index", name)[:10]

        if not records:
            return "NONE"

        records_str += "\n\n".join([f"{i + 1}. {record['name']}" for i, record in enumerate(records)])
        result = FULL_TEXT_FULL_NAME_SEARCH_TEMPLATE.format(
            top_10_matches=records_str
        )

        return result

# Import required modules and initalize COM Store

In [32]:
import nest_asyncio
import asyncio

nest_asyncio.apply()

In [33]:
import sys
import logging

# logging.basicConfig(stream=sys.stdout, level=logging.INFO)
logging.getLogger().addHandler(logging.StreamHandler(sys.stdout))
logging.getLogger().setLevel(logging.INFO)

In [35]:
# from llama_index.core import Settings
# from llama_index.graph_stores.neo4j import Neo4jPropertyGraphStore
# from llama_index.core import PropertyGraphIndex
# from llama_index.core.graph_stores.types import EntityNode, Relation
# from llama_index.core.schema import MetadataMode
from google.colab import userdata
import google.generativeai as genai

genai.configure = getattr(genai.configure, "func", genai.configure)

# Initialize Neo4j connection
com_store =  COMStore(
    username="neo4j",
    password=userdata.get("NEO4J_PASSWORD_KG"),
    url=userdata.get("NEO4J_URI_KG"),
)

ERROR:neo4j.pool:Unable to retrieve routing information


Unable to retrieve routing information
Unable to retrieve routing information
Unable to retrieve routing information


ERROR:neo4j.pool:Unable to retrieve routing information


Unable to retrieve routing information
Unable to retrieve routing information
Unable to retrieve routing information


ERROR:neo4j.pool:Unable to retrieve routing information


Unable to retrieve routing information
Unable to retrieve routing information
Unable to retrieve routing information


... (4 outputs left out)

Transaction failed and will be retried in 1.100613477509983s (Unable to retrieve routing information)
Transaction failed and will be retried in 1.100613477509983s (Unable to retrieve routing information)
Transaction failed and will be retried in 1.100613477509983s (Unable to retrieve routing information)


ERROR:neo4j.pool:Unable to retrieve routing information


Unable to retrieve routing information
Unable to retrieve routing information
Unable to retrieve routing information


Transaction failed and will be retried in 1.6602725719900144s (Unable to retrieve routing information)
Transaction failed and will be retried in 1.6602725719900144s (Unable to retrieve routing information)
Transaction failed and will be retried in 1.6602725719900144s (Unable to retrieve routing information)


# Create and set GeminiWithRateLimits class as LLM

In [6]:
from typing import (
    TYPE_CHECKING,
    Any,
    Dict,
    Generator,
    List,
    Optional,
    Sequence,
    Union,
    cast,
)
from llama_index.llms.gemini.utils import (
    chat_from_gemini_response,
    chat_message_to_gemini,
    completion_from_gemini_response,
)
from llama_index.llms.gemini import Gemini
from llama_index.core.base.llms.types import (
    ChatMessage,
    ChatResponse,
    CompletionResponse
)
from llama_index.core.bridge.pydantic import Field
from llama_index.core.constants import DEFAULT_NUM_OUTPUTS, DEFAULT_TEMPERATURE
from llama_index.core.callbacks import CallbackManager
from llama_index.core.instrumentation.events.llm import (
    LLMPredictEndEvent,
    LLMPredictStartEvent,
    LLMStructuredPredictInProgressEvent,
    LLMStructuredPredictEndEvent,
    LLMStructuredPredictStartEvent,
)
from llama_index.core.base.llms.types import (
    ChatMessage,
    ChatResponse,
    ChatResponseAsyncGen,
    ChatResponseGen,
    CompletionResponse,
    CompletionResponseAsyncGen,
    CompletionResponseGen,
    LLMMetadata,
    MessageRole,
)
from llama_index.core.bridge.pydantic import (
    BaseModel,
    WithJsonSchema,
    Field,
    field_validator,
    model_validator,
    ConfigDict,
    ValidationError,
)
from llama_index.core.types import Model
from llama_index.core.prompts import BasePromptTemplate, PromptTemplate
from llama_index.core.llms.callbacks import llm_chat_callback, llm_completion_callback
from aiolimiter import AsyncLimiter
import google.generativeai as genai
import llama_index.core.instrumentation as instrument

dispatcher = instrument.get_dispatcher(__name__)

GEMINI_MODELS = (
    "models/gemini-2.0-flash",
    "models/gemini-2.0-flash-lite",
    "models/gemini-2.0-flash-thinking",
    "models/gemini-2.0-flash-thinking-exp-01-21",
    "models/gemini-2.0-flash-lite-preview-02-05",
    "models/gemini-2.0-pro-exp-02-05",
    "models/gemini-1.5-flash",
    "models/gemini-1.5-flash-8b",
    "models/gemini-1.0-pro",
)
from llama_index.core.utilities.gemini_utils import (
    merge_neighboring_same_role_messages,
)

class GeminiWithRateLimits(Gemini):
    rate_limit: int = Field(
        default=10,
        description="The number of allowed calls per period."
    )
    period: int = Field(
        default=60,
        description="The period in seconds for the rate limit."
    )
    limiter: AsyncLimiter = Field(
        default=None,
        exclude=True,
        description="The async rate limiter instance."
    )
    def __init__(
        self,
        rate_limit: Optional[int] = 10,  # Default rate limit
        period: Optional[int] = 60,  # Default period
        **kwargs: Any,
    ):
        """Creates a new Gemini model interface."""
        super().__init__(
            **kwargs,
        )
        self.rate_limit = rate_limit
        self.period = period
        self.limiter = AsyncLimiter(self.rate_limit, self.period)

    @llm_chat_callback()
    async def achat(
        self, messages: Sequence[ChatMessage], **kwargs: Any
    ) -> ChatResponse:
        request_options = self._request_options or kwargs.pop("request_options", None)
        merged_messages = merge_neighboring_same_role_messages(messages)
        *history, next_msg = map(chat_message_to_gemini, merged_messages)
        chat = self._model.start_chat(history=history)

        async with self.limiter:
            response = await chat.send_message_async(
                next_msg, request_options=request_options, **kwargs
            )
            return chat_from_gemini_response(response)

    @llm_completion_callback()
    async def acomplete(
        self, prompt: str, formatted: bool = False, **kwargs: Any
    ) -> CompletionResponse:
        request_options = self._request_options or kwargs.pop("request_options", None)
        async with self.limiter:
            result = await self._model.generate_content_async(
                prompt, request_options=request_options, **kwargs
            )
            return completion_from_gemini_response(result)

    @llm_completion_callback()
    def astream_complete(
        self, prompt: str, formatted: bool = False, **kwargs: Any
    ) -> CompletionResponseAsyncGen:
        request_options = self._request_options or kwargs.pop("request_options", None)

        async def gen():
            async with self.limiter:
              it = await self._model.generate_content_async(
                  prompt, stream=True, request_options=request_options, **kwargs
              )
              async for r in it:
                  yield completion_from_gemini_response(r)

        return gen()

    @llm_chat_callback()
    async def astream_chat(
        self, messages: Sequence[ChatMessage], **kwargs: Any
    ) -> ChatResponseAsyncGen:
        request_options = self._request_options or kwargs.pop("request_options", None)
        merged_messages = merge_neighboring_same_role_messages(messages)
        *history, next_msg = map(chat_message_to_gemini, merged_messages)
        chat = self._model.start_chat(history=history)
        async with self.limiter:
          response = await chat.send_message_async(
              next_msg, stream=True, request_options=request_options, **kwargs
          )

          async def gen() -> ChatResponseAsyncGen:
              content = ""
              existing_tool_calls = []
              async for r in response:
                  top_candidate = r.candidates[0]
                  content_delta = top_candidate.content.parts[0].text
                  content += content_delta
                  llama_resp = chat_from_gemini_response(r)
                  existing_tool_calls.extend(
                      llama_resp.message.additional_kwargs.get("tool_calls", [])
                  )
                  llama_resp.delta = content_delta
                  llama_resp.message.content = content
                  llama_resp.message.additional_kwargs["tool_calls"] = existing_tool_calls
                  yield llama_resp

          return gen()

    @dispatcher.span
    async def astructured_predict(
        self, *args: Any, llm_kwargs: Optional[Dict[str, Any]] = None, **kwargs: Any
    ) -> BaseModel:
        """Structured predict."""
        llm_kwargs = llm_kwargs or {}
        all_kwargs = {**llm_kwargs, **kwargs}

        if self._is_function_call_model:
            llm_kwargs["tool_choice"] = (
                "required"
                if "tool_choice" not in all_kwargs
                else all_kwargs["tool_choice"]
            )
        # by default structured prediction uses function calling to extract structured outputs
        # here we force tool_choice to be required
        async with self.limiter:
          return await super().astructured_predict(*args, llm_kwargs=llm_kwargs, **kwargs)

    @dispatcher.span
    async def astream_structured_predict(
        self, *args: Any, llm_kwargs: Optional[Dict[str, Any]] = None, **kwargs: Any
    ) -> Generator[Union[Model, List[Model]], None, None]:
        """Stream structured predict."""
        llm_kwargs = llm_kwargs or {}
        all_kwargs = {**llm_kwargs, **kwargs}

        if self._is_function_call_model:
            llm_kwargs["tool_choice"] = (
                "required"
                if "tool_choice" not in all_kwargs
                else all_kwargs["tool_choice"]
            )
        # by default structured prediction uses function calling to extract structured outputs
        # here we force tool_choice to be required
        async with self.limiter:
          return await super().astream_structured_predict(
              *args, llm_kwargs=llm_kwargs, **kwargs
          )

In [7]:
Settings.llm = GeminiWithRateLimits(
    # model="models/gemini-1.5-flash-latest",
    api_key=userdata.get("GOOGLE_API_KEY"),
    temperature=0.7,
    rate_limit=15,
    period=60
)

In [8]:
# Settings.llm = Gemini(
#     # model="models/gemini-1.5-flash-latest",
#     api_key=userdata.get("GOOGLE_API_KEY"),
#     temperature=1,
#     # rate_limit=5,
#     # period=60
# )

In [9]:
Settings.llm

GeminiWithRateLimits(callback_manager=<llama_index.core.callbacks.base.CallbackManager object at 0x7c2621da8410>, system_prompt=None, messages_to_prompt=<function messages_to_prompt at 0x7c26282fdda0>, completion_to_prompt=<function default_completion_to_prompt at 0x7c26281cd800>, output_parser=None, pydantic_program_mode=<PydanticProgramMode.DEFAULT: 'default'>, query_wrapper_prompt=None, model='models/gemini-2.0-flash', temperature=0.7, max_tokens=8192, generate_kwargs={}, rate_limit=15, period=60, limiter=<AsyncLimiter(max_rate=15, time_period=60) at 0x7c2621d62dd0 [level: 0.000000, waiters: 0]>)

# Set embedding models

In [10]:
from llama_index.embeddings.gemini import GeminiEmbedding
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

In [11]:
# Use for Gemini Embeddings
# Settings.embed_model = GeminiEmbedding(
#     embed_batch_size = 25,
#     model_name = "models/text-embedding-004",
#     api_key = userdata.get("GOOGLE_API_KEY")
# )

In [12]:
# Use for HuggingFace Embeddings
Settings.embed_model = HuggingFaceEmbedding(
    model_name = "mixedbread-ai/mxbai-embed-large-v1",
    embed_batch_size = 25,
)

INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: mixedbread-ai/mxbai-embed-large-v1


Load pretrained SentenceTransformer: mixedbread-ai/mxbai-embed-large-v1


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/266 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/114k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

... (5 outputs left out)

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

INFO:sentence_transformers.SentenceTransformer:2 prompts are loaded, with the keys: ['query', 'text']


2 prompts are loaded, with the keys: ['query', 'text']


In [13]:
print(Settings.embed_model)

model_name='mixedbread-ai/mxbai-embed-large-v1' embed_batch_size=25 callback_manager=<llama_index.core.callbacks.base.CallbackManager object at 0x7c2621da8410> num_workers=None max_length=512 normalize=True query_instruction=None text_instruction=None cache_folder=None


# Use `inspect` to extract MODULES, CLASSES and METHODS in the user specified python module

## Main Code

In [ ]:
"""
ReflEx (Reflection and Extraction) Module

This module provides utilities to extract entities and relationships from Python modules and classes, such as:
- Modules
- Classes
- Methods
- Inheritance
- Imports
"""

from typing import Any, Dict, List, Set, Tuple
import inspect
import importlib
import re
import subprocess
import traceback
import ast
import os
import pkgutil

class Entity:
    """
    Module-agnostic entity class.
    """
    def __init__(self, name: str, label: str, properties: Dict):
        self.name = name
        self.label = label
        self.properties = properties

    def __repr__(self):
        return f"{self.label} ({self.name})"

class Relationship:
    """
    Module-agnostic relation class.
    """
    def __init__(self, from_entity: Entity, to_entity: Entity, label: str, properties: Dict):
        self.from_entity = from_entity
        self.to_entity = to_entity
        self.label = label
        self.properties = properties

    def __repr__(self):
        return f"{self.from_entity} --{self.label}--> {self.to_entity}"

def get_import_statements(source_code: str) -> str:
    """
    Extracts all import statements from the source code, including full
    statements, handling different types, multi-line imports, and 'as' clauses.

    Args:
        source_code (str): The source code of an entity.

    Returns:
        str: All import statements in one clean string block
    """
    try:
      tree = ast.parse(source_code)
      imports = []

      for node in ast.walk(tree):
          if isinstance(node, ast.Import):
              # Handles: import os, sys
              import_statement = "import " + ", ".join(alias.name + (f" as {alias.asname}" if alias.asname else "") for alias in node.names)
              imports.append(import_statement)
          elif isinstance(node, ast.ImportFrom):
              # Handles: from datetime import date, datetime as dt
              names = ", ".join(alias.name + (f" as {alias.asname}" if alias.asname else "") for alias in node.names)
              module = node.module or ''
              level = node.level
              if level > 0:
                module = "." * level + module
              import_statement = f"from {module} import {names}"  #Handles relative imports as well
              imports.append(import_statement)

      return '\n'.join(imports)
    except:
      return ""

def get_import_as_aliases(source_code: str, current_file_path: str) -> List[Tuple[str, str]]:
    """
    Extracts all import ... as ... statements and returns a list of tuples.
    Each tuple contains the full path to the entity and its alias.
    It uses the current_file_path to resolve relative imports to absolute paths.

    Args:
        source_code (str): The source code of an entity.
        current_file_path (str): Full Path of the entity whose source code is being processed

    Returns:
        List[Tuple[str, str]]: List of tuples containing the full path to the entity and its alias.
    """
    try:
      tree = ast.parse(source_code)
      aliases = []

      for node in ast.walk(tree):
          if isinstance(node, ast.Import):
              for alias in node.names:
                  if alias.asname:
                      aliases.append((alias.name, alias.asname))
          elif isinstance(node, ast.ImportFrom):
              module = node.module or ''
              level = node.level
              if level > 0:
                  # Resolve relative import
                  package_path = current_file_path
                  for _ in range(level):
                      package_path = os.path.dirname(package_path)
                      if not package_path:
                          #Reached top level
                          module = "." * level + module
                          break # or raise ValueError("Relative import goes beyond top-level package")

                  if package_path:
                    module = os.path.basename(package_path) + "." + module
                  #If relative import goes beyond current module
                  else:
                    module = "." * level + module
              for alias in node.names:
                  if alias.asname:
                      full_path = f"{module}.{alias.name}" if module else alias.name
                      aliases.append((full_path, alias.asname))
      return aliases
    except:
      return []

def get_unused_imports(module_name: str, imported_names: Set[str]) -> Set[str]:
    """
    Get the list of unused imports in a module using flake8, returning the full paths.

    Args:
        module_name (str): The name of the module.
        imported_names (Set[str]): The set of fully qualified imported names.

    Returns:
        Set[str]: The set of unused imports (full paths).
    """
    try:
        # Get the full file path of the module
        module = importlib.import_module(module_name)
        module_file_path = module.__file__
        print(f"Checking unused imports for {module_name} at {module_file_path}...")

        # Run flake8 and capture output
        process = subprocess.run(
            ["python", "-m", "flake8", "--select", "F401", module_file_path ],
            capture_output=True,
            text=True,
            check=True,
            shell=True,
        )

        output = process.stdout
        unused_names = list()

        # Parse the output for unused import errors (F401), extracting the simple names
        for line in output.splitlines():
            match = re.search(r":\d+:(\d+): F401 '(.*)' imported but unused", line)
            if match:
                unused_names.append(match.group(2))

        # Handle aliases and from-imports statements
        for i in range(len(unused_names)):
            name = unused_names[i].strip()
            if " as " in name:
                unused_names[i] = name.split(" as ")[0]
            if name.startswith("from "):
                unused_names[i] = name.split("from ")[1]
            if " import " in name:
                # Import this unused name to get the fully qualified name
                split_name = name.split(" import ")
                module_name = split_name[0]
                imported_name = split_name[1]
                imported_module = importlib.import_module(module_name)
                imported_entity = getattr(imported_module, imported_name)
                unused_names[i] = get_fully_qualified_name(imported_entity)

        unused_imports = set()
        for full_name in imported_names:
            # if any(name == full_name or full_name.startswith(f"{name}.") for name in unused_names):
            #     unused_imports.add(full_name)
            if any(full_name.endswith(name) for name in unused_names):
                 unused_imports.add(full_name)

        return unused_imports
    except Exception as e:
        print(f"Error getting unused imports for {module_name}: {e}")
        print(traceback.format_exc())
        return set()

def get_fully_qualified_name(cls):
    """
    Get the fully qualified name of a class.

    Args:
        cls: The class object.

    Returns:
        str: The fully qualified name of the class.
    """
    return f"{cls.__module__}.{cls.__qualname__}"

def safe_get_source(obj) -> str:
    """
    Retrieve the source code of an object or fallback to its representation.

    Args:
        obj: The object to retrieve the source code for.

    Returns:
        str: The source code or representation of the object.
    """
    try:
        return inspect.getsource(obj)
    except (OSError, TypeError):
        return f"# Representation:\n{repr(obj)}"

def process_member(
    name: str,
    obj: Any,
    root_module_path: str,
    parent_entity,
    processed_paths: Set[str],
    other_relations: Dict[str, List[Tuple]] = None,
) -> Tuple[List[Entity], List[Relationship]]:
    """
    Process individual members of a class or module.

    Args:
        name (str): Name of the member.
        obj (Any): Member object.
        root_module_path (str): The root module path.
        parent_entity: Parent entity.
        processed_paths (Set[str]): Set of processed paths.
        other_relations (Dict[str, List[Tuple]], optional): Dictionary to store other relations.

    Returns:
        Tuple[List[Entity], List[Relationship]]: List of entities and list of relations.
    """
    entities, relations = [], []

    if inspect.ismodule(obj) or inspect.isclass(obj):
        obj_path = None

        if inspect.ismodule(obj):
            obj_path = obj.__name__
        elif inspect.isclass(obj):
            obj_path = get_fully_qualified_name(obj)

        if not obj_path.startswith(root_module_path + "."):
            if parent_entity and parent_entity.label in {"ROOT_MODULE", "MODULE"}:
                other_relations["IMPORTS"].add((parent_entity.name, obj_path))
            return [], []

        if obj_path in processed_paths:
            if parent_entity and parent_entity.label in {"ROOT_MODULE", "MODULE"}:
                other_relations["IMPORTS"].add((parent_entity.name, obj_path))
            return [], []

        if not obj_path.startswith(parent_entity.name + "."):
            if parent_entity and parent_entity.label in {"ROOT_MODULE", "MODULE"}:
                other_relations["IMPORTS"].add((parent_entity.name, obj_path))
            return [], []

        sub_entities, sub_relations = traverse_recursive(
            obj, root_module_path, processed_paths, other_relations
        )
        entities.extend(sub_entities)
        relations.extend(sub_relations)

        relation_type = "HAS_SUBMODULE" if inspect.ismodule(obj) else "HAS_CLASS"
        relations.append(Relationship(parent_entity, entities[0], relation_type, {}))

    elif inspect.isfunction(obj) and parent_entity.label in {"ROOT_MODULE", "MODULE"}:
        obj_path = get_fully_qualified_name(obj)

        if not obj_path.startswith(root_module_path + "."):
            if parent_entity and parent_entity.label in {"ROOT_MODULE", "MODULE"}:
                other_relations["IMPORTS"].add((parent_entity.name, obj_path))
            return [], []

        if obj_path in processed_paths:
            if parent_entity and parent_entity.label in {"ROOT_MODULE", "MODULE"}:
                other_relations["IMPORTS"].add((parent_entity.name, obj_path))
            return [], []

        elif not obj_path.startswith(parent_entity.name + "."):
            if parent_entity and parent_entity.label in {"ROOT_MODULE", "MODULE"}:
                other_relations["IMPORTS"].add((parent_entity.name, obj_path))
            return [], []

        else:
            processed_paths.add(obj_path)

        source_code = safe_get_source(obj)
        function_entity = Entity(
            obj_path,
            "FUNCTION",
            {
                "source": source_code,
                "__name__": obj.__name__,
                "signature": str(inspect.signature(obj))
            }
        )
        entities.append(function_entity)
        if parent_entity:
            relations.append(Relationship(parent_entity, function_entity, "HAS_FUNCTION", {}))

    elif parent_entity.label == "CLASS" and (inspect.ismethod(obj) or inspect.isfunction(obj)):
        obj_path = get_fully_qualified_name(obj)

        if not obj_path.startswith(root_module_path + ".") or not obj_path.startswith(parent_entity.name + "."):
            return [], []

        if obj_path in processed_paths:
            return [], []
        else:
            processed_paths.add(obj_path)

        source_code = safe_get_source(obj)
        method_entity = Entity(
            obj_path,
            "METHOD",
            {
                "source": source_code,
                "__name__": obj.__name__,
                "signature": str(inspect.signature(obj))
            }
        )
        entities.append(method_entity)
        if parent_entity:
            relations.append(Relationship(parent_entity, method_entity, "HAS_METHOD", {}))

    return entities, relations

def traverse_recursive(
    obj: Any,
    root_module_path: str,
    processed_paths: Set[str],
    other_relations: Dict[str, List[Tuple]] = None,
) -> Tuple[List[Entity], List[Relationship]]:
    """
    Recursively traverse an object (module or class).

    Args:
        obj (Any): The object to traverse.
        root_module_path (str): The root module path.
        processed_paths (Set[str]): Set of processed paths.
        other_relations (Dict[str, List[Tuple]], optional): Dictionary to store other relations.

    Returns:
        Tuple[List[Entity], List[Relation]]: List of entities and list of relations.
    """
    entities, relations = [], []
    obj_path = None
    entity_type = "MODULE" if inspect.ismodule(obj) else "CLASS"
    if entity_type == "MODULE":
        obj_path = obj.__name__
    elif entity_type == "CLASS":
        obj_path = get_fully_qualified_name(obj)

    if not obj_path.startswith(root_module_path + "."):
        return [], []

    if obj_path in processed_paths:
        return [], []

    else:
        processed_paths.add(obj_path)

    source = safe_get_source(obj)
    properties = {
        "source": source,
        "__name__": obj.__name__,
    }

    if inspect.ismodule(obj):
        properties["import_statements"] = get_import_statements(source)
        properties["import_as_aliases"] = get_import_as_aliases(source, obj_path)

    entity = Entity(obj_path, entity_type, properties)
    entities.append(entity)

    if inspect.isclass(obj):
        for base in obj.__bases__:
            if base.__name__ not in ["object", "type"]:
                base_path = f"{base.__module__}.{base.__qualname__}"
                # Check if the following fix works
                if base_path == obj_path:
                    continue
                other_relations["INHERITS"].add((obj_path, base_path))

    # Get all sub-modules using `pkgutil` and NOT `inspect` and then process them
    if inspect.ismodule(obj):
        try:
          path = [obj.__path__[0]]
        except:
          try:
            path = [obj.__file__]
          except:
            path = None
        if path:
          submodules = [name for _, name, _ in pkgutil.iter_modules(path)]
          for submodule_name in submodules:
            try:
              submodule_path = f"{obj.__name__}.{submodule_name}"
              sub_entities, sub_relations = process_member(
                  submodule_name,
                  importlib.import_module(submodule_path),
                  root_module_path,
                  entity,
                  processed_paths,
                  other_relations
              )
              entities.extend(sub_entities)
              relations.extend(sub_relations)
            except:
              continue

    # Process sub-classes
    for name, member in inspect.getmembers(obj, inspect.isclass):
        sub_entities, sub_relations = process_member(
            name, member, root_module_path, entity, processed_paths, other_relations
        )
        entities.extend(sub_entities)
        relations.extend(sub_relations)

    # Process functions
    for name, member in inspect.getmembers(obj, inspect.isfunction):
        sub_entities, sub_relations = process_member(
            name, member, root_module_path, entity, processed_paths, other_relations
        )
        entities.extend(sub_entities)
        relations.extend(sub_relations)

    # Process methods
    for name, member in inspect.getmembers(obj, inspect.ismethod):
        sub_entities, sub_relations = process_member(
            name, member, root_module_path, entity, processed_paths, other_relations
        )
        entities.extend(sub_entities)
        relations.extend(sub_relations)

    return entities, relations

def process_module(module_name: str):
    """
    Process a module or object recursively.

    Args:
        module_name (str): Module name or object.

    Returns:
        Tuple[List[Entity], List[Relation], Dict]: List of entities, list of relations, and other relations.
    """
    entities, relations = [], []
    processed_paths = set()
    other_relations = {"INHERITS": set(), "IMPORTS": set(), "EXPOSES": set()}

    module_object = importlib.import_module(module_name)

    source = safe_get_source(module_object)
    all_import_statements = get_import_statements(source)
    import_as_aliases = get_import_as_aliases(source, module_name)
    properties = {
        "source": source,
        "__name__": module_object.__name__,
        "import_statements": all_import_statements,
        "import_as_aliases": import_as_aliases
    }

    root_entity = Entity(module_name, "ROOT_MODULE", properties)
    entities.append(root_entity)

    # Get all sub-modules using `pkgutil` and NOT `inspect` and then process them
    try:
      path = [module_object.__path__[0]]
    except:
      try:
        path = [module_object.__file__]
      except:
        path = None
    if path:
      submodules = [name for _, name, _ in pkgutil.iter_modules(path)]
      for submodule_name in submodules:
        try:
          submodule_path = f"{module_object.__name__}.{submodule_name}"
          sub_entities, sub_relations = process_member(
              submodule_name,
              importlib.import_module(submodule_path),
              module_name,
              root_entity,
              processed_paths,
              other_relations
          )
          entities.extend(sub_entities)
          relations.extend(sub_relations)
        except:
          continue

    # Process sub-classes
    for name, member in inspect.getmembers(module_object, inspect.isclass):
        sub_entities, sub_relations = process_member(
            name, member, module_name, root_entity, processed_paths, other_relations
        )
        entities.extend(sub_entities)
        relations.extend(sub_relations)

    # Process functions
    for name, member in inspect.getmembers(module_object, inspect.isfunction):
        sub_entities, sub_relations = process_member(
            name, member, module_name, root_entity, processed_paths, other_relations
        )
        entities.extend(sub_entities)
        relations.extend(sub_relations)

    # Process methods
    for name, member in inspect.getmembers(module_object, inspect.ismethod):
        sub_entities, sub_relations = process_member(
            name, member, module_name, root_entity, processed_paths, other_relations
        )
        entities.extend(sub_entities)
        relations.extend(sub_relations)

    # Convert IMPORTS to EXPOSES for unused imports
    imported_relations = other_relations["IMPORTS"]
    updated_imported_relations = set()
    module_to_imports = {}

    for from_entity, to_entity in imported_relations:
        full_module_path = from_entity

        if full_module_path not in module_to_imports:
            module_to_imports[full_module_path] = set()

        module_to_imports[full_module_path].add(to_entity)

    for module, imports in module_to_imports.items():
        unused_imports = get_unused_imports(module, imports)
        used_imports = set(imports) - set(unused_imports)

        for unused_import in unused_imports:
            other_relations["EXPOSES"].add((module, unused_import))

        for used_import in used_imports:
            updated_imported_relations.add((module, used_import))

    other_relations["IMPORTS"] = updated_imported_relations

    cleaned_exposed_relations = set()
    cleaned_imports_relations = set()

    # Clean up IMPORTS relations
    for from_entity, to_entity in other_relations["IMPORTS"]:
        to_entity_ancestor_path = ".".join(to_entity.split(".")[:-1])
        if to_entity_ancestor_path == from_entity:
            continue
        elif to_entity.startswith(module_name):
            other_relations["EXPOSES"].add((from_entity, to_entity))
        else:
            cleaned_imports_relations.add((from_entity, to_entity))

    # Clean up EXPOSED relations
    for from_entity, to_entity in other_relations["EXPOSES"]:
        if not to_entity.startswith(from_entity):
            continue
        cleaned_exposed_relations.add((from_entity, to_entity))

    # Deduplicate entities and relations
    entities = list(set(entities))
    relations = list(set(relations))

    # Clean up Alias Imports
    for entity in entities:
        if entity.label not in ("ROOT_MODULE", "MODULE"):
            continue

        cleaned_import_aliases = []
        for key, value in entity.properties["import_as_aliases"]:
            if (entity, key) in other_relations["EXPOSES"] or (entity, key) in other_relations["IMPORTS"]:
                cleaned_import_aliases.append((key, value))

        entity.properties["import_as_aliases"] = cleaned_import_aliases

    other_relations["EXPOSES"] = list(cleaned_exposed_relations)
    other_relations["IMPORTS"] = list(cleaned_imports_relations)
    other_relations["INHERITS"] = list(other_relations["INHERITS"])

    print(f"Processed {len(entities)} entities and {len(relations)} relations in {module_name}.")

    return entities, relations, other_relations

## Try a demo module

In [ ]:
!pip install -q emoji

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 590.6/590.6 kB 6.0 MB/s eta 0:00:00


In [ ]:
entities, relations, other_relations = process_module("codecs")

Checking unused imports for codecs at /usr/lib/python3.11/codecs.py...
Processed 99 entities and 98 relations in codecs.


### Check entities processed

In [ ]:
def print_modules(entities):
  for entity in entities:
    if entity.label == "MODULE":
      print(entity.label, entity.name)

def print_classes(entities):
  for entity in entities:
    if entity.label == "CLASS":
      print(entity.label, entity.name)

def print_methods(entities):
  for entity in entities:
    if entity.label == "METHOD":
      print(entity.label, entity.name)

def print_functions(entities):
  for entity in entities:
    if entity.label == "FUNCTION":
      print(entity.label, entity.name)

In [ ]:
print_modules(entities)

MODULE concurrent.futures._base
MODULE concurrent.futures.thread
MODULE concurrent.futures.process
MODULE concurrent.futures


In [ ]:
print_classes(entities)

CLASS concurrent.futures._base.BrokenExecutor
CLASS concurrent.futures._base._Waiter
CLASS concurrent.futures.process.BrokenProcessPool
CLASS concurrent.futures.thread.BrokenThreadPool
CLASS concurrent.futures.thread.ThreadPoolExecutor
CLASS concurrent.futures._base.Future
CLASS concurrent.futures.process._SafeQueue
CLASS concurrent.futures.process._RemoteTraceback
CLASS concurrent.futures._base._AcquireFutures
CLASS concurrent.futures._base._AsCompletedWaiter
CLASS concurrent.futures._base.CancelledError
CLASS concurrent.futures.process._ExecutorManagerThread
CLASS concurrent.futures.process._WorkItem
CLASS concurrent.futures.process.ProcessPoolExecutor
CLASS concurrent.futures._base.Error
CLASS concurrent.futures._base.DoneAndNotDoneFutures
CLASS concurrent.futures._base.InvalidStateError
CLASS concurrent.futures._base._FirstCompletedWaiter
CLASS concurrent.futures.process._ExceptionWithTraceback
CLASS concurrent.futures._base.Executor
CLASS concurrent.futures._base._AllCompletedWait

In [ ]:
print_methods(entities)

METHOD concurrent.futures.thread.ThreadPoolExecutor._adjust_thread_count
METHOD concurrent.futures._base.Future.set_exception
METHOD concurrent.futures.process._ExecutorManagerThread.is_shutting_down
METHOD concurrent.futures._base.Future.add_done_callback
METHOD concurrent.futures.thread._WorkItem.run
METHOD concurrent.futures.process._ExecutorManagerThread.__init__
METHOD concurrent.futures.process._RemoteTraceback.__init__
METHOD concurrent.futures.process._ExecutorManagerThread.join_executor_internals
METHOD concurrent.futures._base.Future.done
METHOD concurrent.futures.process._ExceptionWithTraceback.__init__
METHOD concurrent.futures.process._ExecutorManagerThread.run
METHOD concurrent.futures.process._ExecutorManagerThread.wait_result_broken_or_wakeup
METHOD concurrent.futures.process._ExecutorManagerThread.process_result_item
METHOD concurrent.futures.process._ExecutorManagerThread.shutdown_workers
METHOD concurrent.futures._base.Future.set_running_or_notify_cancel
METHOD concu

In [ ]:
print_functions(entities)

FUNCTION concurrent.futures.process._sendback_result
FUNCTION concurrent.futures.process._python_exit
FUNCTION concurrent.futures.process._check_system_limits
FUNCTION concurrent.futures.process._process_chunk
FUNCTION concurrent.futures.thread._worker
FUNCTION concurrent.futures.thread._python_exit
FUNCTION concurrent.futures._base._result_or_cancel
FUNCTION concurrent.futures._base.wait
FUNCTION concurrent.futures._base._yield_finished_futures
FUNCTION concurrent.futures.process._rebuild_exc
FUNCTION concurrent.futures._base.as_completed
FUNCTION concurrent.futures.process._process_worker
FUNCTION concurrent.futures.process._chain_from_iterable_of_lists
FUNCTION concurrent.futures.process._get_chunks
FUNCTION concurrent.futures._base._create_and_install_waiters


In [ ]:
other_relations["INHERITS"]

[('importlib.abc.ExecutionLoader', 'importlib.abc.InspectLoader'),
 ('importlib.resources.simple.TraversableReader',
  'importlib.resources.abc.TraversableResources'),
 ('importlib.abc.SourceLoader', 'importlib.abc.ExecutionLoader'),
 ('importlib.resources.abc.Traversable', 'typing.Protocol'),
 ('importlib.resources._adapters.CompatibilityFiles.ChildPath',
  'importlib.resources.abc.Traversable'),
 ('importlib.metadata.PackageNotFoundError', 'builtins.ModuleNotFoundError'),
 ('importlib.metadata.MetadataPathFinder',
  'importlib.metadata.DistributionFinder'),
 ('importlib.metadata.DistributionFinder', 'importlib.abc.MetaPathFinder'),
 ('importlib.resources.simple.ResourceHandle',
  'importlib.resources.abc.Traversable'),
 ('importlib.resources.readers.ZipReader',
  'importlib.resources.abc.TraversableResources'),
 ('importlib.metadata._collections.FreezableDefaultDict',
  'collections.defaultdict'),
 ('importlib.abc.FileLoader', 'importlib.abc.ResourceLoader'),
 ('importlib.resources.r

In [ ]:
other_relations["IMPORTS"]

[('concurrent.futures', 'builtins.TimeoutError'),
 ('concurrent.futures._base', 'builtins.TimeoutError'),
 ('concurrent.futures.process', 'multiprocessing.queues.Queue'),
 ('concurrent.futures.process', 'traceback.format_exception'),
 ('concurrent.futures.process', 'functools.partial')]

In [ ]:
other_relations["EXPOSES"]

[('concurrent.futures', 'concurrent.futures._base.wait'),
 ('concurrent.futures', 'concurrent.futures._base.CancelledError'),
 ('concurrent.futures', 'concurrent.futures._base.as_completed'),
 ('concurrent.futures', 'concurrent.futures.process.ProcessPoolExecutor'),
 ('concurrent.futures', 'concurrent.futures._base.Future'),
 ('concurrent.futures', 'concurrent.futures._base.BrokenExecutor'),
 ('concurrent.futures', 'concurrent.futures._base.Executor'),
 ('concurrent.futures', 'concurrent.futures.thread.ThreadPoolExecutor')]

### Insert the entities and relations into Neo4j

In [ ]:
com_store.upsert_to_neo4j("codecs", entities, relations, other_relations)

# Upload all standard python modules

In [ ]:
standard_modules = ['string', 're', 'difflib', 'textwrap', 'unicodedata', 'stringprep', 'readline', 'rlcompleter', 'struct', 'codecs', 'datetime', 'zoneinfo', 'calendar', 'collections', 'heapq', 'bisect', 'array', 'weakref', 'types', 'copy', 'pprint', 'reprlib', 'enum', 'graphlib', 'numbers', 'math', 'cmath', 'decimal', 'fractions', 'random', 'statistics', 'itertools', 'functools', 'operator', 'pathlib', 'stat', 'filecmp', 'tempfile', 'glob', 'fnmatch', 'linecache', 'shutil', 'pickle', 'copyreg', 'shelve', 'marshal', 'dbm', 'sqlite3', 'zlib', 'gzip', 'bz2', 'lzma', 'zipfile', 'tarfile', 'csv', 'configparser', 'tomllib', 'netrc', 'plistlib', 'hashlib', 'hmac', 'secrets', 'os', 'io', 'time', 'logging', 'platform', 'errno', 'ctypes', 'argparse', 'optparse', 'getpass', 'fileinput', 'curses', 'threading', 'multiprocessing', 'concurrent', 'subprocess', 'sched', 'queue', 'contextvars', 'asyncio', 'socket', 'ssl', 'select', 'selectors', 'signal', 'mmap', 'email', 'json', 'mailbox', 'mimetypes', 'base64', 'binascii', 'quopri', 'html', 'xml', 'pyexpat', 'webbrowser', 'wsgiref', 'urllib', 'http', 'ftplib', 'poplib', 'imaplib', 'smtplib', 'uuid', 'socketserver', 'xmlrpc', 'ipaddress', 'wave', 'colorsys', 'gettext', 'locale', 'turtle', 'cmd', 'shlex', 'tkinter', 'typing', 'pydoc', 'doctest', 'unittest', 'test', 'bdb', 'faulthandler', 'pdb', 'profile', 'timeit', 'trace', 'tracemalloc', 'venv', 'zipapp', 'sys', 'sysconfig', 'builtins', 'warnings', 'dataclasses', 'contextlib', 'abc', 'atexit', 'traceback', 'gc', 'inspect', 'site', 'code', 'codeop', 'zipimport', 'pkgutil', 'modulefinder', 'runpy', 'importlib', 'ast', 'symtable', 'token', 'keyword', 'tokenize', 'tabnanny', 'pyclbr', 'py_compile', 'compileall', 'dis', 'pickletools', 'getopt']

In [ ]:
from tqdm import tqdm

for standard_module in tqdm(standard_modules, desc="Processing Modules"):
    com_store.structured_query(
        """
        MATCH (n {name: $module_name})-[r:HAS_CLASS|HAS_FUNCTION|HAS_METHOD|HAS_SUBMODULE*]->(e)
        DETACH DELETE n, e
        """,
        {
            "module_name": standard_module
        }
    )
    entities, relations, other_relations = process_module(standard_module)
    com_store.upsert_to_neo4j(standard_module, entities, relations, other_relations)

Processing Modules:   0%|          | 0/163 [00:00<?, ?it/s]

Checking unused imports for string at /usr/lib/python3.11/string.py...
Processed 20 entities and 19 relations in string.


Processing Modules:   1%|          | 1/163 [00:20<55:58, 20.73s/it]

Checking unused imports for re._constants at /usr/lib/python3.11/re/_constants.py...
Checking unused imports for re._parser at /usr/lib/python3.11/re/_parser.py...
Checking unused imports for re._compiler at /usr/lib/python3.11/re/_compiler.py...
Processed 87 entities and 86 relations in re.


Processing Modules:   1%|          | 2/163 [00:42<56:40, 21.12s/it]

Checking unused imports for difflib at /usr/lib/python3.11/difflib.py...
Processed 49 entities and 48 relations in difflib.


... (334 outputs left out)

Checking unused imports for pickletools at /usr/lib/python3.11/pickletools.py...
Processed 39 entities and 38 relations in pickletools.


Processing Modules:  99%|█████████▉| 162/163 [1:01:52<00:23, 23.04s/it]

Checking unused imports for getopt at /usr/lib/python3.11/getopt.py...
Processed 10 entities and 9 relations in getopt.


Processing Modules: 100%|██████████| 163/163 [1:02:15<00:00, 22.92s/it]


# Simple Multi Agent Workflow

In [36]:
#@title Create necessary tools

# from llama_index.tools.duckduckgo import DuckDuckGoSearchToolSpec
from llama_index.core.tools import FunctionTool, ToolMetadata

def next_top_k_root_modules(top_k: int = 10):
  return com_store.next_entities(from_="ROOT_MODULE", top_k=top_k)

def next_top_k_children(top_k: int = 10):
  return com_store.next_entities(from_="CHILDREN", top_k=top_k)

# ddgs_tools = DuckDuckGoSearchToolSpec().to_tool_list()
retrieve_root_modules_tool = FunctionTool.from_defaults(
    fn=com_store.retrieve_relevant_root_modules,
    description="Retrieve relevant root modules based on the query, using cosine similarity search"
)
retrieve_children_tool = FunctionTool.from_defaults(
    fn=com_store.retrieve_relevant_children_in_subtree,
    description="Retrieve relevant immediate children of the current node based on the query, using cosine similarity search"
)
get_entity_info = FunctionTool.from_defaults(
    fn=com_store.get_entity_info,
    description="""
    Get the information for the parent module of the entity from the code knowledge graph.
    Includes Name, Source Code, Summary, Imported Entities, Full Path and Shortest Call Path of the requested entity from it's root module.
    """
)
get_full_path_name = FunctionTool.from_defaults(
    fn=com_store.get_full_path_name,
    description="Get the full path name of the entity. Uses direct cypher query for deterministic response. If not found, uses Full-Text Search for Max Top 3 Matches."
)
get_next_top_k_root_modules = FunctionTool.from_defaults(
    fn=next_top_k_root_modules,
    description="""
    Get the next top k root modules. Use when you want the next set of root modules, from a previous call to `retrieve_root_modules_tool`.
    """
)
get_next_top_k_children = FunctionTool.from_defaults(
    fn=next_top_k_children,
    description="""
    Get the next top k children. Use when you want the next set of children, from a previous call to `retrieve_children_tool`.
    """
)

all_root_tools = [retrieve_root_modules_tool, get_next_top_k_root_modules]
all_code_tools = [get_entity_info, get_full_path_name]
all_search_tools = [retrieve_children_tool, get_next_top_k_children]

In [37]:
#@title Create Agent Workflow
from llama_index.core.agent.workflow import (
    AgentWorkflow,
    FunctionAgent,
    AgentInput,
    AgentOutput,
    ToolCall,
    ToolCallResult,
    AgentStream,
)

root_system_prompt = """
You are a powerful agent designed to answer coding-related questions using tools that interact with a code knowledge graph. This knowledge graph contains information about code components, their relationships, and functionalities.

You have access to the following tools, with their descriptions and relevant information:

1. **retrieve_root_modules_tool**: This tool retrieves the root modules relevant to the user's query. Root modules are the main modules that contain the primary functions or classes related to the query.
    - Parameters: query (string, required), top_k (int, default=10), offset (int, default=0)
    - Returns: List of root modules with their descriptions and relevant information.
    - Example Output:
        ```
        Relevant Root Modules:

        1. module1: summary1
        2. module2: summary2
        ...
        ```
    **Notes**:
    - Do not modify top_k and offset unless ABSOLUTELY necessary. The default values are set for optimal results.
    - You may use this tool a maximum of 10 times in a row, changing the offset by increments of top_k to get the next set of root modules. Keep the other parameters, inlcuding the query, the exact same.
    - You may use this tool like so (multiple times), if the results you received were unsatisfactory.
    - There will be VERY FEW SCENARIOS where top_k would be changed from the default value. Only do so if the user asks for it, or in similar scenarios.

2. **get_next_top_k_root_modules**: This tool retrieves the next set of root modules relevant to the user's query. Use this to get the next top k root modules, from a previous call to `retrieve_root_modules_tool`.
    - Parameters: top_k (int, default=10)
    - Returns: List of root modules with their descriptions and relevant information.
    **Notes**:
    - PLEASE USE THIS TOOL WHEN YOU WANT TO GET THE NEXT SET OF ROOT MODULES, FROM A PREVIOUS CALL TO `retrieve_root_modules_tool`, ESPECIALLY IF THE PREVIOUS CALL TO `retrieve_root_modules_tool` WAS UNSATISFACTORY.
    - This tool will tell you when all the root modules have been explored. In this case, you are to use `retrieve_root_modules_tool` with a modified query.
    - You may keep using this tool till the above case is encountered.

**Note**:
- Make sure you provide queries that are as comprehensive as possible for the above tools to work effectively. (As they use cosine similarity to match queries to code components)
- For example, a query designed as '_ in python' will yield pathetic results. Instead, depending on the tool used, a query like 'Find a functionality that ____ in python' or 'Python ____ that ____' will yield much better results. Just make sure they are long and comprehensive enough.
- When using `retrieve_root_modules_tool`, design the query as 'python module designed for ____' or 'python library that ____' for better results. Again, make sure they are comprehensive enough. Reference the type of functionality required as well as some related stuff.

Here's how to approach each query:

**1. Initial Assessment:**

*   *Always* begin by using `retrieve_root_modules_tool` with the user's query. This will give you an overview of the relevant code components in the knowledge graph.
*   If the retrieved root modules are unsatisfactory, use the `get_next_top_k_root_modules` tool to get the next set of root modules.
*   Follow the above mentioned notes.

**2. Handling Different Query Types:**

*   **If the query is COMPLEX (requiring multiple components, planning, graph traversal, includes MOST IMPORTANTLY finding a way to do something):**
    - After using the tools and gaining enough knowledge, dissect the user's query into simpler, focused sub-queries.

*  **If the query is SIMPLE (directly answerable by a single component, includes exploration, discovery, learning, understanding):**
    - Proceed to the next step.

**3. Deciding type of query:**

*   **If the query involves finding specific entities and maybe code generation:**
    *   Handoff to the SEARCH AGENT to find the specific entity that satisfies the user's query, specifying what you need, along with the full path from the root module to the entity or the root module's name itself.
    *   The SEARCH AGENT may return multiple satisfying entities, but you should focus on the most relevant one.
    *   If SEARCH AGENT wasn't able to find a satisfying entity (due to restrictions on continuous tool usage), you may ask it continue a max of 2 times, after which ask the user for more specific details or whether they allow more exploration.
    *   DO NOT ASK THE SEARCH AGENT TO SEARCH FOR CODE EXAMPLES OR SNIPPETS.
    *   If the query involves code generation, you can ask the specialized CODE AGENT to generate the required code snippets, by providing the necessary details.
    *   The above steps may occur multiple times if the query involves multiple components or functionalities.
    *   Once you have all the required code snippets, you can combine them to generate the final code snippet that fulfills the overall functionality desired in the original user's query. Then, return the final code snippet to the user.

*   **If the query involves simple exploration and leans towards discovering, learning and understanding stuff:**
    *   Handoff to SEARCH AGENT to explore the immediate children of a specific satisfying entity in the code knowledge graph.
    *   Summarize the findings and provide the information to the user.
    *   You may use this tool again if the user asks for more specific details.

**Important Notes**:
- If the user's query involves exploration-related activities, code generation is not required.
- However, if you are dealing with (or want to) code generation or anything coding related, ALWAYS handoff the specialized CODE AGENT to do so. Do not attempt to generate or modify or help with code snippets yourself, directly.
- In the above case, before handing off to the CODE AGENT, PLEASE MAKE SURE TO TRY YOUR BEST TO FIND THE RELEVANT FUNCTIONALITIES using the SEARCH AGENT and the tools at your disposal. Once you are satisfied or have tried enough, handoff to CODE AGENT in the above case.
MAKE SURE YOU ANALYZE your previous interaction, and then make use of the tools and SEARCH AGENT to find the required entities. Please don't just skip to handing off to CODE AGENT directly. Do not assume it to do the tasks of SEARCH AGENT.
- Once you get all the required code snippets from the specialized CODE AGENT, you can combine them to generate the final code snippet that fulfills the overall functionality desired in the original user's query. Then, return the final code snippet to the user.
- MOST IMPORTANTLY, whenever you interact with the user, explain your findings in detail, and then IF NEEDED, ask for more specific details to provide a more accurate response. This will help in understanding the user's requirements better and providing more relevant answers.
- FINALLY, for all user queries, you are to make the most use of the tools at your disposal and agents you can handoff to, to get the most amount of context and information possible. Don't just skip to directly answering the user's query.

Key Points:

* Use the retrieved modules to first understand if the query can be full-filled with all that is on that module and the relationships. Only then should you generate a plan.*
* If there are a small number of functions, do not try to create atomic tasks to do so.*
"""

code_system_prompt = """
You are a powerful Code Agent responsible for generating Python code snippets.  You receive instructions from the Root Agent, specifying entities and desired functionalities. Your primary goal is to produce accurate, functional, and executable code.

**Crucially, before generating *any* code, you *must* use your tools to gather detailed information about the provided entities.**  Assumptions are strictly forbidden.

**Your Tools:**

1.  **`get_entity_info(entity_name)`**:  *MANDATORY*. Retrieves comprehensive information about a code entity (function, class, method).  This includes:
    *   Parent Module Name & Full Path
    *   Shortest Call Path (for imports)
    *   Summary Description
    *   **Source Code of the Parent Module** (essential for analysis)

2.  **`get_full_path_name(entity_name)`**: Utility tool to get the full path of an entity if needed. Useful for verification or if `get_entity_info` doesn't work as expected initially.

**Workflow (Strictly Enforced):**

1.  **Receive Instructions**: The Root Agent provides you with:
    *   List of entity full paths.
    *   Functionality descriptions for each entity.
    *   Overall code structure plan.

2.  **MANDATORY Information Retrieval**: For *each* entity:
    *   **ALWAYS** use `get_entity_info(entity_name)` with the entity's full path. *Do this first, every time.*
    *   *Optionally*, use `get_full_path_name(entity_name)` to verify the full path if needed.
    *   *Remember*, if `get_entity_info(entity_name)` does not return the expected information, use `get_full_path_name(entity_name)` to get the full path and then attempt to use `get_entity_info(entity_name)` again.

3.  **Thorough Source Code Analysis**:  Carefully examine the `source_code` from `get_entity_info` to understand:
    *   **Entity Signature**: Parameters (names, types), return type.
    *   **Docstring**: Purpose, usage, parameter details, return value information.
    *   **Return Statements**: Confirm return type.
    *   **Non-Built-in Types**: If parameters or return types are custom classes, analyze their source code (especially `__init__`) to understand instantiation and usage. Focus on first-level information unless deeper analysis is explicitly required and permitted.

4.  **Generate Python Code**: Based on the *information gathered from your tools* and Root Agent's instructions:
    *   Include necessary imports using `shortest_call_path`.
    *   Use entities correctly with appropriate parameters (types, values).
    *   Handle return values properly.
    *   Implement the requested functionality accurately.
    *   Ensure code is executable and well-formatted.

5.  **Return Code Snippet**: Provide the generated, executable code to the Root Agent.

**Key Principles - Remember These!**

*   **NO ASSUMPTIONS**:  *Never* assume anything.  *Always* use `get_entity_info` to get details.
*   **TOOL USAGE IS MANDATORY**: `get_entity_info` is *not optional*.  Use it for every entity *before* coding.
*   **SOURCE CODE IS TRUTH**: Analyze the `source_code` from `get_entity_info` to guide code generation.
*   **ACCURACY & FUNCTIONALITY**:  Prioritize generating correct, working code that meets the Root Agent's requirements.
*   **USE SHORTEST CALL PATHS**: For clean import statements.
*   **HANDOFF TO ROOT AGENT AFTER GENERATING THE CODE**: Always handoff to the Root Agent after generating the code.

If you are unsure or need clarification, ask the Root Agent. Your goal is to generate high-quality code based on verified information, not guesswork.
"""

search_system_prompt = """
You are a powerful agent designed to explore and find relevant code components based on user queries. You have access to a code knowledge graph that contains information about code components, their relationships, and functionalities.
You do not have to generate examples of code snippets. Just find the relevant code components based on the user's query.

You have access to the following tools, with their descriptions and relevant information:

1. **retrieve_children_tool**: This tool retrieves the IMMEDIATE children of a given entity in the code knowledge graph.
    - Parameters: query (string, required), parent_name (string, required), top_k (int, default=10), offset (int, default=0)
    - Returns: List of all IMMEDIATE children entities with their descriptions and relevant information.
    - Example Output:
        ```
        Relevant Children in Subtree:

        1. child1: summary1
        2. child2: summary2
        ...
        ```
    **Notes**:
    - This tool is used to explore the immediate children of a given entity to explore deeper functionalities. The `query` parameter will only be used to sort the children based on relevance to the query. However, all the IMMEDIATE children will be returned.
    - There will be VERY FEW SCENARIOS where top_k would be changed from the default value. Only do so if the user asks for it, or in similar scenarios.
    - If the returned children are unsatisfactory or you couldn't find what you were looking for, you may use this tool again, with ALL THE SAME PARAMETERS, especially the query.
    Smartly increment the offset based on top_k used. Basically, you may get the next set of children in line, in the above case.
    - Do not call this tool more than 10 times in a row.
    - Please keep using this until you are satisfied with the results.

2. **get_next_top_k_children**: This tool retrieves the next set of children. Use this to get the next top k children, from a previous call to `retrieve_children_tool`.
    - Parameters: top_k (int, default=10)
    - Returns: List of top_k IMMEDIATE children entities with their descriptions and relevant information.
    **Notes**:
    - PLEASE USE THIS TOOL WHEN YOU WANT TO GET THE NEXT SET OF CHILDREN, FROM A PREVIOUS CALL TO `retrieve_children_tool`, ESPECIALLY IF THE PREVIOUS CALL TO `retrieve_children_tool` WAS UNSATISFACTORY.
    - This tool will tell you when all the children have been explored. In this case, you are to use `retrieve_children_tool` with a modified query.
    - You may keep using this tool till the above case is encountered.

**Note**:
- Make sure you provide queries that are as comprehensive as possible for the above tools to work effectively. (As they use cosine similarity to match queries to code components)
- For example, a query designed as '_ in python' will yield pathetic results. Instead, depending on the tool used, a query like 'Find a functionality that ____ in python' or 'Python ____ that ____' will yield much better results. Just make sure they are long and comprehensive enough.
- YOU HAVE TO THINK, REASON AND ANALYZE THE ROOT AGENT'S QUERY (HANDOFF QUERY). As the tools make use of cosine/semantic similarity search, you have to create an appropriate query that doesn't BLINDLY mention stuff from the ROOT AGENT's query, instead makes use of the available context to create context rich queries.

Queries will MOSTLY involve finding a functionality, regardless of the type, as well as the ROOT MODULE, whose tree has to be explored, using the tools provided.

Here's how to approach each query:

**1. Initial Assessment:**

- Use the `retrieve_children_tool` with the user's query to explore the immediate children of a given entity in the code knowledge graph, sorted by relevance to the query.
- If the returned children are unsatisfactory, use the `get_next_top_k_children` tool to get the next set of children.
- If after 7 tries, you still can't find what you were looking for, handoff to the ROOT AGENT with your findings, in detail, for further assistance. ROOT AGENT may ask you to continue exploring, if needed.

**2. Subsequent Steps:**

- Evaluate the returned children to understand their functionalities and relevance to the user's query.
- Unless specified, your job would be to find a FUNCTION or a METHOD that fulfills the user's query. CLASS entities may also satisfy the query, but they are less common. Regardless, if they look VERY relevant, they should be considered.
- Once found, handoff to the ROOT AGENT, with your findings, mentioning the FULL PATH from the ROOT MODULE to the entity found, as well as the entity's summary and other relevant information, depending on the query.
- Regardless whether a satisfactory entity is found or not, do not call the tool more than 5 times in a row. In such cases, handoff to the ROOT AGENT with your findings, in detail, for further assistance. ROOT AGENT may ask you to continue exploring, if needed.

**Important Notes**:
- Remember, your job is to return the FULL PATH of entity/entities (from the ROOT MODULE) satisfying the user's query.
- Don't try to look for a specific entity (that may be mentioned by the ROOT AGENT) again inside a ROOT MODULE (as the tools assigned are not built for that). Instead, look for entities deeper in the subtree of the specific entity.
This is because the ROOT AGENT may be guiding your search for more specific entities.
- The objective is to find either a single satisfying entity, or a combination of them, that can satisfy the user's query. Don't try to look for examples, or generate them. Just handoff your findings, in detail, to the ROOT AGENT.
- You are to also add accompanying information, such as the entity's summary, to help the ROOT AGENT understand the relevance of the entity to the user's query.
"""

root_agent = FunctionAgent(
    name="RootAgent",
    description="Agent that interacts with user primarily, uses tools to get an idea as to how the KG can help and then get relevant entites that can solve the query together",
    tools=all_root_tools,
    system_prompt=root_system_prompt,
    can_handoff_to=["CodeAgent", "SearchAgent"]
)

code_agent = FunctionAgent(
    name="CodeAgent",
    description="Agent that generates code using requested entities for requested functionalities, after getting all of their important info using tools",
    tools=all_code_tools,
    system_prompt=code_system_prompt,
    can_handoff_to=["RootAgent"]
)

search_agent = FunctionAgent(
    name="SearchAgent",
    description="Agent that explores and finds relevant code components based on user queries",
    tools=all_search_tools,
    system_prompt=search_system_prompt,
    can_handoff_to=["RootAgent"]
)

agent_workflow = AgentWorkflow(
    agents=[root_agent, code_agent, search_agent],
    root_agent=root_agent.name
)

In [38]:
#@title Create a chat wrapper named 'chat_with_agent_workflow' over handler logic
async def chat_with_agent_workflow(user_msg, handler_ctx = None):
  handler = agent_workflow.run(
      ctx=handler_ctx,
      user_msg=user_msg
  )

  current_agent = None
  current_tool_calls = ""

  async for event in handler.stream_events():
      if (
          hasattr(event, "current_agent_name")
          and event.current_agent_name != current_agent
      ):
          current_agent = event.current_agent_name
          print(f"\n{'='*50}")
          print(f"🤖 Agent: {current_agent}")
          print(f"{'='*50}\n")

      # if isinstance(event, AgentStream):
      #     if event.delta:
      #         print(event.delta, end="", flush=True)
      # elif isinstance(event, AgentInput):
      #     print("📥 Input:", event.input)
      elif isinstance(event, AgentOutput):
          if event.response.content:
              print("📤 Output:", event.response.content)
          if event.tool_calls:
              print(
                  "🛠️  Planning to use tools:",
                  [call.tool_name for call in event.tool_calls],
              )
      elif isinstance(event, ToolCallResult):
          print(f"🔧 Tool Result ({event.tool_name}):")
          print(f"  Arguments: {event.tool_kwargs}")
          print(f"  Output: {event.tool_output}")
      elif isinstance(event, ToolCall):
          print(f"🔨 Calling Tool: {event.tool_name}")
          print(f"  With arguments: {event.tool_kwargs}")

  return handler

## Following is the FIRST working example of a basic agent workflow running for requests.api.get (DO NOT TOUCH)


In [ ]:
handler = agent_workflow.run(
    user_msg="I want to retrieve some data from a URL. Suggest a method to do so."
)

In [ ]:
from llama_index.core.agent.workflow import (
    AgentInput,
    AgentOutput,
    ToolCall,
    ToolCallResult,
    AgentStream,
)

current_agent = None
current_tool_calls = ""

async for event in handler.stream_events():
    if (
        hasattr(event, "current_agent_name")
        and event.current_agent_name != current_agent
    ):
        current_agent = event.current_agent_name
        print(f"\n{'='*50}")
        print(f"🤖 Agent: {current_agent}")
        print(f"{'='*50}\n")

    # if isinstance(event, AgentStream):
    #     if event.delta:
    #         print(event.delta, end="", flush=True)
    # elif isinstance(event, AgentInput):
    #     print("📥 Input:", event.input)
    elif isinstance(event, AgentOutput):
        if event.response.content:
            print("📤 Output:", event.response.content)
        if event.tool_calls:
            print(
                "🛠️  Planning to use tools:",
                [call.tool_name for call in event.tool_calls],
            )
    elif isinstance(event, ToolCallResult):
        print(f"🔧 Tool Result ({event.tool_name}):")
        print(f"  Arguments: {event.tool_kwargs}")
        print(f"  Output: {event.tool_output}")
    elif isinstance(event, ToolCall):
        print(f"🔨 Calling Tool: {event.tool_name}")
        print(f"  With arguments: {event.tool_kwargs}")


🤖 Agent: RootAgent

📤 Output: ```
Thought: The current language of the user is: English. I need to use a tool to help me answer the question.
Action: retrieve_relevant_root_modules
Action Input: {"query": "retrieve data from URL"}
```
🛠️  Planning to use tools: ['retrieve_relevant_root_modules']
🔨 Calling Tool: retrieve_relevant_root_modules
  With arguments: {'query': 'retrieve data from URL'}
🔧 Tool Result (retrieve_relevant_root_modules):
  Arguments: {'query': 'retrieve data from URL'}
  Output: Relevant Root Modules:

1. **urllib**: ### Summary of `urllib` Module

#### Overview
The `urllib` module in Python is a package that collects several modules for working with URLs. It provides tools for opening URLs using various protocols and includes functionalities for parsing and manipulating URLs.

#### Sub-Modules

1. **urllib.request**
   - Provides classes and functions for opening and reading URLs. It handles HTTP, FTP, and other protocols.
   - Includes classes like `Request`, `O

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

=== Function Output ===
Relevant Root Modules:

1. **requests**: ### Summary of `requests`

#### Overview
`requests` is a Python HTTP library designed to simplify the process of making HTTP requests. It provides a user-friendly interface for interacting with web services and is widely used due to its ease of use and robust feature set.

#### Key Functionalities
1. **HTTP Requests**: Supports various HTTP methods such as GET, POST, PUT, DELETE, etc.
2. **Session Management**: Manages cookies, headers, and other settings across multiple requests using sessions.
3. **Authentication**: Supports basic, proxy, and digest authentication.
4. **Proxy Support**: Handles HTTP and SOCKS proxies.
5. **SSL Verification**: Performs secure connections with automatic verification.
6. **Content Handling**: Automatically decompresses and decodes content.
7. **File Uploads**: Facilitates multi-part file uploads.
8. **Timeouts**: Allows specifying connection and read timeouts.
9. **Streaming**: Supports do

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

=== Function Output ===
Relevant Children in Subtree:

1. **requests.check_compatibility**: The `check_compatibility` function is responsible for verifying the compatibility of the `requests` library with its dependencies, specifically `urllib3`, `chardet`, and `charset_normalizer`. It ensures that the installed versions of these libraries meet the minimum requirements for `requests` to function correctly.

The function first parses the version string of `urllib3`, handling cases where the version might be abbreviated (e.g., "1.16"). It then asserts that `urllib3` is at least version 1.21.1. This check ensures that `requests` is using a compatible version of the underlying HTTP library.

Next, the function checks for the presence and version of either `chardet` or `charset_normalizer`, which are used for character encoding detection. If `chardet` is present, it verifies that its version is between 3.0.2 (inclusive) and 6.0.0 (exclusive). If `chardet` is not found, it checks for `charse

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

=== Function Output ===
Relevant Children in Subtree:

1. **requests.check_compatibility**: The `check_compatibility` function is responsible for verifying the compatibility of the `requests` library with its dependencies, specifically `urllib3`, `chardet`, and `charset_normalizer`. It ensures that the installed versions of these libraries meet the minimum requirements for `requests` to function correctly.

The function first parses the version string of `urllib3`, handling cases where the version might be abbreviated (e.g., "1.16"). It then asserts that `urllib3` is at least version 1.21.1. This check ensures that `requests` is using a compatible version of the underlying HTTP library.

Next, the function checks for the presence and version of either `chardet` or `charset_normalizer`, which are used for character encoding detection. If `chardet` is present, it verifies that its version is between 3.0.2 (inclusive) and 6.0.0 (exclusive). If `chardet` is not found, it checks for `charse

## Following is the second working example of a basic agent workflow running for requests.api.get (DO NOT TOUCH)


In [ ]:
handler = agent_workflow.run(
    user_msg="I want to retrieve some data from a URL. Suggest a method to do so."
)

In [ ]:
from llama_index.core.agent.workflow import (
    AgentInput,
    AgentOutput,
    ToolCall,
    ToolCallResult,
    AgentStream,
)

current_agent = None
current_tool_calls = ""

async for event in handler.stream_events():
    if (
        hasattr(event, "current_agent_name")
        and event.current_agent_name != current_agent
    ):
        current_agent = event.current_agent_name
        print(f"\n{'='*50}")
        print(f"🤖 Agent: {current_agent}")
        print(f"{'='*50}\n")

    # if isinstance(event, AgentStream):
    #     if event.delta:
    #         print(event.delta, end="", flush=True)
    # elif isinstance(event, AgentInput):
    #     print("📥 Input:", event.input)
    elif isinstance(event, AgentOutput):
        if event.response.content:
            print("📤 Output:", event.response.content)
        if event.tool_calls:
            print(
                "🛠️  Planning to use tools:",
                [call.tool_name for call in event.tool_calls],
            )
    elif isinstance(event, ToolCallResult):
        print(f"🔧 Tool Result ({event.tool_name}):")
        print(f"  Arguments: {event.tool_kwargs}")
        print(f"  Output: {event.tool_output}")
    elif isinstance(event, ToolCall):
        print(f"🔨 Calling Tool: {event.tool_name}")
        print(f"  With arguments: {event.tool_kwargs}")


🤖 Agent: RootAgent

📤 Output: ```
Thought: The current language of the user is: English. I need to use a tool to help me answer the question.
Action: retrieve_relevant_root_modules
Action Input: {"query": "retrieve data from URL"}
```
🛠️  Planning to use tools: ['retrieve_relevant_root_modules']
🔨 Calling Tool: retrieve_relevant_root_modules
  With arguments: {'query': 'retrieve data from URL'}
🔧 Tool Result (retrieve_relevant_root_modules):
  Arguments: {'query': 'retrieve data from URL'}
  Output: Relevant Root Modules:

1. **urllib**: ### Summary of `urllib` Module

#### Overview
The `urllib` module in Python is a package that collects several modules for working with URLs. It provides tools for opening URLs using various protocols and includes functionalities for parsing and manipulating URLs.

#### Sub-Modules

1. **urllib.request**
   - Provides classes and functions for opening and reading URLs. It handles HTTP, FTP, and other protocols.
   - Includes classes like `Request`, `O

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

=== Function Output ===
Relevant Root Modules:

1. **requests**: ### Summary of `requests`

#### Overview
`requests` is a Python HTTP library designed to simplify the process of making HTTP requests. It provides a user-friendly interface for interacting with web services and is widely used due to its ease of use and robust feature set.

#### Key Functionalities
1. **HTTP Requests**: Supports various HTTP methods such as GET, POST, PUT, DELETE, etc.
2. **Session Management**: Manages cookies, headers, and other settings across multiple requests using sessions.
3. **Authentication**: Supports basic, proxy, and digest authentication.
4. **Proxy Support**: Handles HTTP and SOCKS proxies.
5. **SSL Verification**: Performs secure connections with automatic verification.
6. **Content Handling**: Automatically decompresses and decodes content.
7. **File Uploads**: Facilitates multi-part file uploads.
8. **Timeouts**: Allows specifying connection and read timeouts.
9. **Streaming**: Supports do

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

=== Function Output ===
Relevant Children in Subtree:

1. **requests.check_compatibility**: The `check_compatibility` function is responsible for verifying the compatibility of the `requests` library with its dependencies, specifically `urllib3`, `chardet`, and `charset_normalizer`. It ensures that the installed versions of these libraries meet the minimum requirements for `requests` to function correctly.

The function first parses the version string of `urllib3`, handling cases where the version might be abbreviated (e.g., "1.16"). It then asserts that `urllib3` is at least version 1.21.1. This check ensures that `requests` is using a compatible version of the underlying HTTP library.

Next, the function checks for the presence and version of either `chardet` or `charset_normalizer`, which are used for character encoding detection. If `chardet` is present, it verifies that its version is between 3.0.2 (inclusive) and 6.0.0 (exclusive). If `chardet` is not found, it checks for `charse

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

=== Function Output ===
Relevant Children in Subtree:

1. **requests.check_compatibility**: The `check_compatibility` function is responsible for verifying the compatibility of the `requests` library with its dependencies, specifically `urllib3`, `chardet`, and `charset_normalizer`. It ensures that the installed versions of these libraries meet the minimum requirements for `requests` to function correctly.

The function first parses the version string of `urllib3`, handling cases where the version might be abbreviated (e.g., "1.16"). It then asserts that `urllib3` is at least version 1.21.1. This check ensures that `requests` is using a compatible version of the underlying HTTP library.

Next, the function checks for the presence and version of either `chardet` or `charset_normalizer`, which are used for character encoding detection. If `chardet` is present, it verifies that its version is between 3.0.2 (inclusive) and 6.0.0 (exclusive). If `chardet` is not found, it checks for `charse

## Test for aerosandbox

In [ ]:
#@title What is aerosandbox?

await chat_with_agent_workflow("What can I use for aerodynamics related stuff in python?")

NameError: name 'agent_workflow' is not defined

In [ ]:
#@title Understanding aerosandbox

await chat_with_agent_workflow("What can aerosandbox be used for? How can use it to calculate drag over a particular aircraft design?")


🤖 Agent: RootAgent

🛠️  Planning to use tools: ['retrieve_relevant_root_modules']
🔨 Calling Tool: retrieve_relevant_root_modules
  With arguments: {'query': 'aerosandbox python library for aircraft design and aerodynamic analysis', 'top_k': 5.0}


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

🔧 Tool Result (retrieve_relevant_root_modules):
  Arguments: {'query': 'aerosandbox python library for aircraft design and aerodynamic analysis', 'top_k': 5.0}
  Output: Relevant Root Modules:

1. **aerosandbox** (similarity score: 0.9435531497001648):### Summary of `aerosandbox`

#### Overview
`aerosandbox` is a comprehensive Python package designed for aircraft design and optimization, integrating advanced automatic differentiation and differentiable aerospace physics models. It provides a wide range of tools for aerodynamics, structures, propulsion, and more, making it a powerful suite for researchers and engineers.

#### Key Sub-Modules

1. **`aerosandbox.geometry`**
   - **Purpose**: Handles geometric entities such as polygons, wings, fuselages, airfoils, and propulsors.
   - **Key Features**: Polygon handling, wing and fuselage analysis, airfoil analysis, propulsor modeling, and mesh manipulation.

2. **`aerosandbox.weights`**
   - **Purpose**: Manages mass properties of aircraft

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

🔧 Tool Result (retrieve_relevant_children_in_subtree):
  Arguments: {'query': 'python geometry module for defining aircraft geometry', 'parent_name': 'aerosandbox.geometry', 'top_k': 20.0}
  Output: Relevant Children in Subtree:

1. **aerosandbox.geometry.wing** (similarity score: 0.9018300771713257, relevant labels: ['MODULE']):
### Summary of `aerosandbox.geometry.wing`

#### Overview
The `aerosandbox.geometry.wing` module defines a comprehensive framework for creating and analyzing wing geometries in aerospace simulations. It provides a detailed representation of wings, including their cross-sections (`WingXSec`) and control surfaces, and offers a wide range of methods for calculating various aerodynamic properties and visualizing the wing.

#### Key Functionalities
1. **Wing Definition**: The `Wing` class defines a wing composed of multiple `WingXSec` objects, representing cross-sections along the wing span. It supports both symmetric and asymmetric wings and allows for the specifi

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

... (3 outputs left out)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

🔧 Tool Result (retrieve_relevant_children_in_subtree):
  Arguments: {'query': 'python function for calculating aerodynamic drag using vortex lattice method', 'parent_name': 'aerosandbox', 'top_k': 20.0}
  Output: Relevant Children in Subtree:

1. **aerosandbox.docs** (similarity score: 0.8502241969108582, relevant labels: ['FUNCTION']):
The aerosandbox.docs entity is designed to open the AeroSandbox documentation in a web browser. It consists of a single method, `docs()`, which accomplishes this task. The method uses the `webbrowser` module to open a new tab with the specified URL.

Key functionalities include:
- Redirecting the user to the AeroSandbox documentation on GitHub.
- Providing easy access to the documentation without requiring manual navigation.

The `docs()` method contributes to the overall usability of the AeroSandbox library by offering a straightforward way to access its documentation. This enhances user experience and facilitates learning and troubleshooting.

2. **ae

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

🔧 Tool Result (retrieve_relevant_children_in_subtree):
  Arguments: {'query': 'python module for calculating aerodynamic drag using vortex lattice method', 'parent_name': 'aerosandbox.aerodynamics.aero_3D', 'top_k': 20.0}
  Output: Relevant Children in Subtree:

1. **aerosandbox.aerodynamics.aero_3D.aero_buildup_submodels** (similarity score: 0.8877478837966919, relevant labels: ['MODULE']):
### Comprehensive Summary of `aerosandbox.aerodynamics.aero_3D.aero_buildup_submodels`

#### Overview
The `aerosandbox.aerodynamics.aero_3D.aero_buildup_submodels` module is part of the `aerosandbox` library and focuses on providing utilities for estimating aerodynamic characteristics of fuselages, particularly in transonic flow regimes. This module includes functions designed to model various aspects of fuselage aerodynamics, such as critical Mach numbers, crossflow lift multipliers, base drag coefficients, and form factors.

#### Key Components

1. **`fuselage_aerodynamics_utilities` Module**
   

## Test for aerosandbox with multi-turn conversations

In [25]:
handler = await chat_with_agent_workflow("What's aerosandbox?")


🤖 Agent: RootAgent

🛠️  Planning to use tools: ['retrieve_relevant_root_modules']
🔨 Calling Tool: retrieve_relevant_root_modules
  With arguments: {'query': 'What is aerosandbox?', 'top_k': 5.0}


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

🔧 Tool Result (retrieve_relevant_root_modules):
  Arguments: {'query': 'What is aerosandbox?', 'top_k': 5.0}
  Output: Relevant Root Modules:

1. **aerosandbox** (similarity score: 0.8699472546577454):The `aerosandbox` module is a comprehensive toolkit designed for aerospace engineering simulations. It provides a wide range of functionalities, from aerodynamic analysis and structural mechanics to optimization and data visualization.

### Purpose
The primary purpose of this module is to offer a robust framework for simulating and analyzing various aspects of aircraft and spacecraft. It supports both preliminary design studies and detailed simulations.

### Functionality
The module includes sub-modules for different tasks:
- **Optimization**: Simplifies the setup and solution of optimization problems using CasADi and IPOPT.
- **Geometry**: Handles geometric shapes and components, facilitating comprehensive modeling and analysis.
- **Atmosphere**: Models atmospheric conditions like pressu

In [26]:
handler = await chat_with_agent_workflow("""I'd like to know how it can be used for creating and analyzing various aircraft types like biplanes, monoplanes etc.""", handler_ctx=handler.ctx)
# handler = await chat_with_agent_workflow("""I'd like to know how it can be used for creating and analyzing various aircraft types like biplanes, monoplanes etc.""")


🤖 Agent: RootAgent

🛠️  Planning to use tools: ['retrieve_relevant_root_modules']
🔨 Calling Tool: retrieve_relevant_root_modules
  With arguments: {'query': 'aerosandbox aircraft design biplanes monoplanes analysis', 'top_k': 5.0}


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

🔧 Tool Result (retrieve_relevant_root_modules):
  Arguments: {'query': 'aerosandbox aircraft design biplanes monoplanes analysis', 'top_k': 5.0}
  Output: Relevant Root Modules:

1. **aerosandbox** (similarity score: 0.8749605417251587):The `aerosandbox` module is a comprehensive toolkit designed for aerospace engineering simulations. It provides a wide range of functionalities, from aerodynamic analysis and structural mechanics to optimization and data visualization.

### Purpose
The primary purpose of this module is to offer a robust framework for simulating and analyzing various aspects of aircraft and spacecraft. It supports both preliminary design studies and detailed simulations.

### Functionality
The module includes sub-modules for different tasks:
- **Optimization**: Simplifies the setup and solution of optimization problems using CasADi and IPOPT.
- **Geometry**: Handles geometric shapes and components, facilitating comprehensive modeling and analysis.
- **Atmosphere**: Model

In [39]:
handler = await chat_with_agent_workflow("""What can I use for getting data from a URL?""")


🤖 Agent: RootAgent

🛠️  Planning to use tools: ['retrieve_relevant_root_modules']
🔨 Calling Tool: retrieve_relevant_root_modules
  With arguments: {'query': 'python module for getting data from a URL', 'top_k': 5.0}


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

🔧 Tool Result (retrieve_relevant_root_modules):
  Arguments: {'query': 'python module for getting data from a URL', 'top_k': 5.0}
  Output: Relevant Root Modules:

1. **urllib** (similarity score: 0.8754590749740601):The `urllib` module in Python is a collection of modules designed to handle URL operations, including parsing, requesting, and managing errors. It serves as a fundamental tool for web development, providing a range of functionalities that simplify working with URLs and HTTP requests.

### Purpose
The primary purpose of the `urllib` module is to facilitate various aspects of URL handling, from parsing URLs into their components to making HTTP requests and managing errors encountered during these operations. This module is essential for tasks such as web scraping, API interactions, and general network communication.

### Functionality
The `urllib` module consists of several sub-modules, each serving a specific purpose:
- **`urllib.parse`**: Handles URL parsing and reconstruc

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

🔧 Tool Result (retrieve_relevant_children_in_subtree):
  Arguments: {'query': 'function for getting data from a URL', 'parent_name': 'requests', 'top_k': 5.0}
  Output: Relevant Children in Subtree:

1. **requests.api.get** (similarity score: 0.8831849694252014, relevant labels: ['FUNCTION']):
The `requests.api.get` function sends a GET request to the specified URL with optional parameters and keyword arguments. It is used for making HTTP GET requests and returns a `requests.Response` object.

2. **requests.api.options** (similarity score: 0.8494876623153687, relevant labels: ['FUNCTION']):
The `requests.api.options` function sends an HTTP OPTIONS request to the specified URL with optional parameters. It is useful for checking the capabilities of a server.

3. **requests.api.post** (similarity score: 0.8399206399917603, relevant labels: ['FUNCTION']):
The `requests.api.post` function sends a POST request to a specified URL with optional data or JSON payload. It is used to submit data t

In [40]:
# handler = await chat_with_agent_workflow("""Give me a code to experiment with aerosandbox with my biplane configs for drag estimation and analysis""", handler_ctx=handler.ctx)
handler = await chat_with_agent_workflow("""Give me a code to experiment with aerosandbox with my biplane configs for drag estimation and analysis""")


🤖 Agent: RootAgent

📤 Output: Okay, I will start by retrieving relevant root modules for biplane drag estimation and analysis using aerosandbox.
🛠️  Planning to use tools: ['retrieve_relevant_root_modules']
🔨 Calling Tool: retrieve_relevant_root_modules
  With arguments: {'query': 'aerosandbox module for biplane drag estimation and analysis'}


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

🔧 Tool Result (retrieve_relevant_root_modules):
  Arguments: {'query': 'aerosandbox module for biplane drag estimation and analysis'}
  Output: Relevant Root Modules:

1. **aerosandbox** (similarity score: 0.8898682594299316):The `aerosandbox` module is a comprehensive toolkit designed for aerospace engineering simulations. It provides a wide range of functionalities, from aerodynamic analysis and structural mechanics to optimization and data visualization.

### Purpose
The primary purpose of this module is to offer a robust framework for simulating and analyzing various aspects of aircraft and spacecraft. It supports both preliminary design studies and detailed simulations.

### Functionality
The module includes sub-modules for different tasks:
- **Optimization**: Simplifies the setup and solution of optimization problems using CasADi and IPOPT.
- **Geometry**: Handles geometric shapes and components, facilitating comprehensive modeling and analysis.
- **Atmosphere**: Models atmospher

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

🔧 Tool Result (retrieve_relevant_children_in_subtree):
  Arguments: {'query': 'biplane drag estimation and analysis', 'parent_name': 'aerosandbox', 'top_k': 10.0}
  Output: Relevant Children in Subtree:

1. **aerosandbox.dynamics.point_mass.point_2D.speed_gamma.DynamicsPointMass2DSpeedGamma** (similarity score: 0.8372630476951599, relevant labels: ['CLASS']):
The `DynamicsPointMass2DSpeedGamma` class models the dynamics of a point mass in two-dimensional space, where the velocity is parameterized by speed and flight path angle (gamma). This class extends `DynamicsPointMass3DSpeedGammaTrack` and is designed for aerodynamic simulations focusing on simplified aircraft motion.

### Functionality
This class initializes several key state variables such as position (`x_e`, `z_e`), speed, and gamma. It also includes indirect control variables like angle of attack (`alpha`) and direct control variables like forces in the wind axes (`Fx_w`, `Fz_w`). The `state_derivatives` method computes the ti

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

... (15 outputs left out)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

🔧 Tool Result (retrieve_relevant_root_modules):
  Arguments: {'query': 'aerosandbox Airplane class constructor Wing', 'top_k': 5.0}
  Output: Relevant Root Modules:

1. **aerosandbox** (similarity score: 0.8778315782546997):The `aerosandbox` module is a comprehensive toolkit designed for aerospace engineering simulations. It provides a wide range of functionalities, from aerodynamic analysis and structural mechanics to optimization and data visualization.

### Purpose
The primary purpose of this module is to offer a robust framework for simulating and analyzing various aspects of aircraft and spacecraft. It supports both preliminary design studies and detailed simulations.

### Functionality
The module includes sub-modules for different tasks:
- **Optimization**: Simplifies the setup and solution of optimization problems using CasADi and IPOPT.
- **Geometry**: Handles geometric shapes and components, facilitating comprehensive modeling and analysis.
- **Atmosphere**: Models atmospheric

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

🔧 Tool Result (retrieve_relevant_root_modules):
  Arguments: {'query': 'aerosandbox Airplane class arguments', 'top_k': 5.0}
  Output: Relevant Root Modules:

1. **aerosandbox** (similarity score: 0.8506258726119995):The `aerosandbox` module is a comprehensive toolkit designed for aerospace engineering simulations. It provides a wide range of functionalities, from aerodynamic analysis and structural mechanics to optimization and data visualization.

### Purpose
The primary purpose of this module is to offer a robust framework for simulating and analyzing various aspects of aircraft and spacecraft. It supports both preliminary design studies and detailed simulations.

### Functionality
The module includes sub-modules for different tasks:
- **Optimization**: Simplifies the setup and solution of optimization problems using CasADi and IPOPT.
- **Geometry**: Handles geometric shapes and components, facilitating comprehensive modeling and analysis.
- **Atmosphere**: Models atmospheric condit

In [ ]:
# handler = await chat_with_agent_workflow("""Is there a way to dynamically have the code optimize my biplane setup, given some metrics and other things to be kept constant (such as altitude, speed)?""", handler_ctx=handler.ctx)
handler = await chat_with_agent_workflow("""Is there a way to dynamically have the code optimize my biplane setup, given some metrics and other things to be kept constant (such as altitude, speed)?""")


🤖 Agent: RootAgent

📤 Output: Okay, I understand the query. The user wants to dynamically optimize their biplane setup using code, given some metrics and constraints like altitude and speed.

Here's my plan:

1.  **Find relevant modules:** I'll start by using the `retrieve_root_modules_tool` to find Python modules that might be useful for optimization, control systems, or flight dynamics. I'll use a comprehensive query to get the best results.
2.  **Explore modules:** I'll examine the descriptions of the retrieved modules to identify the most promising ones.
3.  **Identify key functionalities:** Once I have a good understanding of the available modules, I'll use the `handoff` tool to ask the `SearchAgent` to find specific entities (functions, classes, etc.) within those modules that can be used for optimization, control, or simulation.
4.  **Code Generation (if needed):** If the user wants code generation, I'll use the `handoff` tool to ask the `CodeAgent` to generate the required cod

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

🔧 Tool Result (retrieve_relevant_root_modules):
  Arguments: {'query': 'python module for flight dynamics, optimization, control systems, and simulation of aircraft, including biplanes, with constraints on altitude and speed', 'top_k': 5.0}
  Output: Relevant Root Modules:

1. **aerosandbox** (similarity score: 0.8872715830802917):### Summary of `aerosandbox`

#### Overview
`aerosandbox` is a comprehensive Python package designed for aircraft design and optimization, integrating advanced automatic differentiation and differentiable aerospace physics models. It provides a wide range of tools for aerodynamics, structures, propulsion, and more, making it a powerful suite for researchers and engineers.

#### Key Sub-Modules

1. **`aerosandbox.geometry`**
   - **Purpose**: Handles geometric entities such as polygons, wings, fuselages, airfoils, and propulsors.
   - **Key Features**: Polygon handling, wing and fuselage analysis, airfoil analysis, propulsor modeling, and mesh manipulation.

2

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

🔧 Tool Result (retrieve_relevant_children_in_subtree):
  Arguments: {'query': 'dynamic optimization of biplane configuration with altitude and speed constraints', 'parent_name': 'aerosandbox.optimization', 'top_k': 10.0}
  Output: Relevant Children in Subtree:

1. **aerosandbox.optimization.opti.Opti** (similarity score: 0.8133896589279175, relevant labels: ['CLASS']):
The `aerosandbox.optimization.opti.Opti` entity is a base class for mathematical optimization in the Aerosandbox framework. It leverages CasADi for symbolic computation and utilizes IPOPT as the backend solver. Key functionalities include initializing variables, constraints, and solving optimization problems.

### Key Members and Their Contributions:

- **`variable`**: Initializes decision variables with options for initial guesses, scales, and bounds. It supports both scalar and vector variables and allows for freezing variables and applying logarithmic transformations.
- **`subject_to`**: Adds equality or inequality co

In [ ]:
!pip install -q aerosandbox

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 40.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.4/73.4 MB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.4/119.4 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.3/20.3 MB 72.6 MB/s eta 0:00:00


In [ ]:
import aerosandbox as asb
import aerosandbox.numpy as np
from aerosandbox.geometry import Airplane, Wing, WingXSec, Airfoil
from aerosandbox.performance import OperatingPoint
from aerosandbox.aerodynamics.aero_3D.lifting_line import LiftingLine

# Define biplane geometry
wing_upper = Wing(
    name="Upper Wing",
    xsecs=[
        WingXSec(
            xyz_le=[0, 0, 1],  # Positioned 1 meter above the origin
            chord=1,
            airfoil=Airfoil("naca0012")
        ),
        WingXSec(
            xyz_le=[0, 5, 1],  # Positioned 1 meter above the origin
            chord=1,
            airfoil=Airfoil("naca0012")
        ),
    ]
)

wing_lower = Wing(
    name="Lower Wing",
    xsecs=[
        WingXSec(
            xyz_le=[0, 0, 0],
            chord=1,
            airfoil=Airfoil("naca0012")
        ),
        WingXSec(
            xyz_le=[0, 5, 0],
            chord=1,
            airfoil=Airfoil("naca0012")
        ),
    ]
)

biplane = Airplane(
    name="Biplane",
    wings=[wing_upper, wing_lower]
)

# Define operating point
op_point = OperatingPoint(
    velocity=100,  # m/s
    alpha=5,  # degrees
    beta=0,  # degrees
)

# Create lifting line analysis
lifting_line_analysis = LiftingLine(
    airplane=biplane,
    op_point=op_point,
    spanwise_resolution=10 # Increased resolution for better accuracy
)

# Run analysis
aero_results = lifting_line_analysis.run()

# Print results
print("Lift Coefficient (CL):", aero_results["CL"])
print("Drag Coefficient (CD):", aero_results["CD"])
print("Lift-to-Drag Ratio (L/D):", aero_results["CL"] / aero_results["CD"])

Lift Coefficient (CL): 0.7435751598729898
Drag Coefficient (CD): 0.035813970224813974
Lift-to-Drag Ratio (L/D): 20.762153852403618


In [ ]:
import aerosandbox as asb
import aerosandbox.numpy as np

# Define the biplane setup (simplified)
wing_chord = 1.0  # meters
wing_span = 10.0  # meters
altitude = 1000.0 # meters
speed = 50.0 # m/s

# Create an optimization environment
opti = asb.Opti()

# Define optimization variables
control_surface_deflection = opti.variable(init_guess=0.0, scale=0.1, lower_bound=-np.pi/6, upper_bound=np.pi/6)  # radians
wing_twist = opti.variable(init_guess=0.0, scale=0.1, lower_bound=-np.pi/12, upper_bound=np.pi/12) # radians


# Define the objective function (maximize lift-to-drag ratio - a placeholder)
def lift_to_drag_ratio(control_surface_deflection, wing_twist, altitude, speed):
    # This is a placeholder - replace with a real aerodynamic model
    lift = 10 + 5*control_surface_deflection + 2*wing_twist - 0.001*altitude + 0.01*speed
    drag = 1 + 2*control_surface_deflection**2 + wing_twist**2 + 0.00001*altitude + 0.0001*speed**2
    return lift / drag

objective = lift_to_drag_ratio(control_surface_deflection, wing_twist, altitude, speed)
opti.maximize(objective)

# Define constraints (altitude and speed - these are just examples)
opti.subject_to(altitude >= 500)
opti.subject_to(altitude <= 2000)
opti.subject_to(speed >= 30)
opti.subject_to(speed <= 70)


# Solve the optimization problem
sol = opti.solve()

# Print the optimized values
if sol:
    print("Optimized control surface deflection:", sol(control_surface_deflection))
    print("Optimized wing twist:", sol(wing_twist))
    print("Objective function value:", sol(objective))
else:
    print("Optimization failed!")

This is Ipopt version 3.14.11, running with linear solver MUMPS 5.4.1.

Number of nonzeros in equality constraint Jacobian...:        0
Number of nonzeros in inequality constraint Jacobian.:        4
Number of nonzeros in Lagrangian Hessian.............:        3

Total number of variables............................:        2
                     variables with only lower bounds:        0
                variables with lower and upper bounds:        0
                     variables with only upper bounds:        0
Total number of equality constraints.................:        0
Total number of inequality constraints...............:        4
        inequality constraints with only lower bounds:        2
   inequality constraints with lower and upper bounds:        0
        inequality constraints with only upper bounds:        2

iter    objective    inf_pr   inf_du lg(mu)  ||d||  lg(rg) alpha_du alpha_pr  ls
   0 -7.5396825e+00 0.00e+00 1.32e-01   0.0 0.00e+00    -  0.00e+00 0.00e+00 